# BioVid Part B — biosignal-only pain intensity classification

Leave-one-subject-out (LOSO) pipeline on the filtered biosignals of the BioVid Heat Pain Database, Part B
(EDA/GSR, ECG and three EMG channels; no video, no metadata).

| Step | What it does | Writes to |
|---|---|---|
| 1 | Index and audit every `*_bio.csv`, exclude corrupt or flat trials, cache the signals per subject | `outputs/features/` |
| 2 | 232 hand-crafted features (61 EDA, 51 ECG, 40 per EMG channel); baseline-similarity features with a full and a leave-one-out BL1 template | `outputs/features/` |
| 3 | Feature hygiene: drop constants, greedy correlation de-duplication (\|r\| > 0.95 on within-subject z-scores) | `outputs/features/` |
| 4 | LOSO with XGBoost and HistGradientBoosting on identical folds, four configurations | `outputs/models/` |
| 5 | Evaluation with subject-level bootstrap CIs and paired model comparison | `outputs/reports/` |
| 6 | PA1 facial-EMG duplication audit | `outputs/reports/` |
| 7 | Figures | `outputs/figures/` |
| 8 | Linear baselines (logistic regression, RBF-SVM) on the same folds | `outputs/models/` |
| 9 | The published Gruss et al. (2015) feature set under the same protocol (optional) | `outputs/features/`, `outputs/models/` |
| 10 | Model and feature-set comparisons with paired subject-level bootstraps | `outputs/reports/`, `outputs/figures/` |
| 11 | Summary of every result needed for the paper | `outputs/reports/summary.md` |

Configuration comes from a `.env` file (see `.env.example`); only `BIOVID_PARTB_DIR` is required.
Step 9 runs only when `BIOVID_PLOS_FEATURES` points to the published feature table.
Run the notebook top to bottom from this folder. Every step reads only files written earlier in the same run.

In [1]:
import json
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))


def env_path(name, default=None):
    value = os.environ.get(name, "").strip()
    if value:
        return Path(value).expanduser().resolve()
    if default is None:
        raise RuntimeError(f"{name} is not set. Copy .env.example to .env and edit it.")
    return Path(default).resolve()


DATASET_DIR = env_path("BIOVID_PARTB_DIR")                       # one sub-folder per subject
OUTPUT_DIR = env_path("BIOVID_OUTPUT_DIR", Path.cwd() / "outputs")
N_JOBS = int(os.environ.get("BIOVID_N_JOBS", "-1"))              # joblib / XGBoost workers, -1 = all cores
SEED = 42
# Optional: Gruss et al. (2015) feature table (Table_Step2_159Features-85Subs-5Levels-z.xlsx); enables Step 9.
_plos = os.environ.get("BIOVID_PLOS_FEATURES", "").strip()
PLOS_FEATURES_FILE = Path(_plos).expanduser().resolve() if _plos else None

assert DATASET_DIR.is_dir(), f"BIOVID_PARTB_DIR does not exist: {DATASET_DIR}"
assert PLOS_FEATURES_FILE is None or PLOS_FEATURES_FILE.is_file(), f"BIOVID_PLOS_FEATURES not found: {PLOS_FEATURES_FILE}"

FEATURES_DIR = OUTPUT_DIR / "features"
CACHE_DIR = FEATURES_DIR / "cache"
MODELS_DIR = OUTPUT_DIR / "models"
REPORTS_DIR = OUTPUT_DIR / "reports"
FIG_DIR = OUTPUT_DIR / "figures"
for d in (CACHE_DIR, MODELS_DIR, REPORTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Layout of the filtered Part B release: 5 channels, 2816 samples per trial (5.5 s at 512 Hz).
CHANNELS = ["gsr", "ecg", "emg_trapezius", "emg_corrugator", "emg_zygomaticus"]
EMG_CHANNELS = {"trapezius": 2, "corrugator": 3, "zygomaticus": 4}   # name -> channel index
SEG_LEN = 2816
FLAT_EPS = 1e-8                                                      # std <= FLAT_EPS means a dead channel

ID_COLS = ["subject_id", "subject_name", "class_id", "class_name", "sample_id", "sample_name"]
CLASS_MAP = {"BL1": 0, "PA1": 1, "PA2": 2, "PA3": 3, "PA4": 4}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}
LABELS = [0, 1, 2, 3, 4]
MODELS = ["xgb", "hgb"]
MODEL_NAMES = {"xgb": "XGBoost", "hgb": "HGB"}
LOO_PREFIX = "loo__"                                                 # leave-one-out similarity columns

# XGBoost is the final shallow configuration from Part A (not tuned on Part B).
# HGB is capacity-matched: depth 3, 300 trees, same learning rate and L2.
XGB_PARAMS = dict(
    objective="multi:softprob", eval_metric="mlogloss", tree_method="hist",
    learning_rate=0.07, max_depth=3, min_child_weight=10, subsample=0.8,
    colsample_bytree=0.6, reg_lambda=2.0, reg_alpha=0.0, n_estimators=300,
    random_state=SEED, n_jobs=N_JOBS,
)
XGB_BINARY_PARAMS = {**XGB_PARAMS, "objective": "binary:logistic", "eval_metric": "logloss"}
HGB_PARAMS = dict(
    loss="log_loss", learning_rate=0.07, max_iter=300, max_depth=3, max_leaf_nodes=15,
    min_samples_leaf=20, l2_regularization=2.0, max_bins=255, early_stopping=False,
    random_state=SEED,
)


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


print("dataset:", DATASET_DIR)
print("outputs:", OUTPUT_DIR)
print("n_jobs :", N_JOBS)
print("published features:", PLOS_FEATURES_FILE or "not set (Step 9 is skipped)")

dataset: D:\BioVid_Full\Bio Vid Part B\biosignals_filtered\biosignals_filtered
outputs: D:\HGB\Final_pipeline\outputs
n_jobs : -1
published features: D:\BioVid_Part_A\PartB_PLOS_ONE_2015_biofeatures\PartB_PLOS_ONE_2015_biofeatures\Table_Step2_159Features-85Subs-5Levels-z.xlsx


## Step 1 — index, audit and cache

Every `*_bio.csv` is read once. A trial is retained only if it has 2816 rows, no NaN and no dead channel.
Retained signals are cached per subject as one `(n_trials, 5, 2816)` float32 array; the full index keeps the
audit columns of every trial, including the excluded ones.

In [2]:
from joblib import Parallel, delayed

FILE_RE = re.compile(r"^(?P<subject>.+)-(?P<cls>BL1|PA[1-4])-(?P<num>\d+)_bio\.csv$", re.IGNORECASE)


def audit_subject(subject_dir):
    """Read every trial of one subject, audit it and cache the clean trials as one array."""
    rows, arrays = [], []
    for path in sorted(subject_dir.glob("*_bio.csv")):
        m = FILE_RE.match(path.name)
        if not m:
            continue
        d = pd.read_csv(path, sep="\t")
        sig = d[CHANNELS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64).T
        t = pd.to_numeric(d["time"], errors="coerce").to_numpy(dtype=np.float64)
        nan_per_ch = np.isnan(sig).sum(axis=1)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")                      # all-NaN channels
            std_per_ch = np.nanstd(sig, axis=1)
            max_abs = float(np.nanmax(np.abs(sig)))
        row = {
            "subject_name": subject_dir.name,
            "class_name": m.group("cls").upper(),
            "sample_num": int(m.group("num")),
            "sample_name": path.name[: -len("_bio.csv")],
            "bio_path": str(path),
            "n_rows": int(len(d)),
            "fs_hz": 1e6 / float(np.median(np.diff(t))),         # time stamps are in microseconds
            "first_time": float(t[0]),
            "last_time": float(t[-1]),
            "max_abs": max_abs,
            "has_nan": bool(nan_per_ch.sum() > 0),
        }
        for c, n_nan, sd in zip(CHANNELS, nan_per_ch, std_per_ch):
            row[f"nan_{c}"] = int(n_nan)
            row[f"std_{c}"] = float(sd)
            row[f"flat_{c}"] = bool(not np.isfinite(sd) or sd <= FLAT_EPS)
        row["keep"] = bool(
            row["n_rows"] == SEG_LEN
            and not row["has_nan"]
            and not any(row[f"flat_{c}"] for c in CHANNELS)
        )
        rows.append(row)
        arrays.append(sig.astype(np.float32) if row["keep"] else None)

    df = pd.DataFrame(rows)
    df["class_id"] = df["class_name"].map(CLASS_MAP)
    order = np.lexsort((df["sample_name"].to_numpy(), df["sample_num"].to_numpy(), df["class_id"].to_numpy()))
    df = df.iloc[order].reset_index(drop=True)
    kept = [arrays[i] for i in order if arrays[i] is not None]
    df["cache_file"] = f"{subject_dir.name}.npy"
    df["cache_row"] = -1
    df.loc[df["keep"], "cache_row"] = np.arange(len(kept))
    np.save(CACHE_DIR / df["cache_file"].iloc[0],
            np.stack(kept) if kept else np.empty((0, len(CHANNELS), SEG_LEN), np.float32))
    return df


subject_dirs = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir())
assert subject_dirs, f"no subject folders under {DATASET_DIR}"

t0 = time.perf_counter()
index = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(audit_subject)(d) for d in subject_dirs), ignore_index=True)
print(f"audited {len(index)} trials from {len(subject_dirs)} subjects in {time.perf_counter() - t0:.1f}s")

index["subject_id"] = index["subject_name"].astype("category").cat.codes.astype(int)
index = index.sort_values(["subject_id", "class_id", "sample_num", "sample_name"]).reset_index(drop=True)
index["sample_id"] = np.arange(len(index))
index = index[ID_COLS + [c for c in index.columns if c not in ID_COLS]]

# ---- integrity of the release ----
assert index["sample_name"].is_unique
assert index["n_rows"].eq(SEG_LEN).all(), index["n_rows"].value_counts().to_dict()
assert index["fs_hz"].between(500, 525).all(), "expected ~512 Hz with microsecond time stamps"
assert index["class_id"].nunique() == 5
assert index.loc[index["keep"], "max_abs"].lt(1e7).all(), "extreme values in a retained trial"

reasons = {"corrupt_or_nan": index["has_nan"], **{f"flat_{c}": index[f"flat_{c}"] for c in CHANNELS}}
print("\nexclusion reasons (a trial can have several):")
for name, mask in reasons.items():
    print(f"  {name:22s} {int(mask.sum()):5d}")
print(f"  {'unique excluded':22s} {int((~index['keep']).sum()):5d}")

meta = index.loc[index["keep"]].drop(columns="keep").reset_index(drop=True)
per_subject = meta.groupby("subject_id").size()

# ---- round trip: three random cached trials must equal their source files ----
rng = np.random.default_rng(SEED)
for k in rng.choice(len(meta), size=3, replace=False):
    r = meta.iloc[k]
    src = pd.read_csv(r["bio_path"], sep="\t")[CHANNELS].to_numpy(dtype=np.float32).T
    cached = np.load(CACHE_DIR / r["cache_file"], mmap_mode="r")[r["cache_row"]]
    assert np.array_equal(src, cached), r["sample_name"]

index.to_parquet(FEATURES_DIR / "index_biosignals_partB.parquet", index=False)
meta.to_parquet(CACHE_DIR / "meta.parquet", index=False)
cache_mb = sum(p.stat().st_size for p in CACHE_DIR.glob("*.npy")) / 1024**2
write_json(CACHE_DIR / "cache_manifest.json", {
    "dataset_dir": str(DATASET_DIR),
    "n_index_rows": int(len(index)),
    "n_cached": int(len(meta)),
    "n_excluded": int((~index["keep"]).sum()),
    "excluded_by_reason": {k: int(v.sum()) for k, v in reasons.items()},
    "channels": CHANNELS,
    "array_shape_per_trial": [len(CHANNELS), SEG_LEN],
    "dtype": "float32",
    "flat_eps": FLAT_EPS,
    "fs_hz_median": float(meta["fs_hz"].median()),
    "per_class_cached": {CLASS_NAMES[int(k)]: int(v) for k, v in meta["class_id"].value_counts().sort_index().items()},
    "cache_mib": round(cache_mb, 1),
})

print("\nretained trials :", len(meta))
print("subjects        :", meta["subject_id"].nunique())
print("per class       :", {CLASS_NAMES[int(k)]: int(v) for k, v in meta["class_id"].value_counts().sort_index().items()})
print("per subject     : min=%d median=%.0f max=%d" % (per_subject.min(), per_subject.median(), per_subject.max()))
print("subjects < 90   :", meta.groupby("subject_name").size().loc[lambda s: s < 90].to_dict())
print("fs_hz           : %.4f" % meta["fs_hz"].median())
print("cache           : %d files, %.0f MiB" % (len(subject_dirs), cache_mb))

audited 8600 trials from 86 subjects in 13.1s

exclusion reasons (a trial can have several):
  corrupt_or_nan             6
  flat_gsr                  51
  flat_ecg                   0
  flat_emg_trapezius         2
  flat_emg_corrugator       57
  flat_emg_zygomaticus      59
  unique excluded          174



retained trials : 8426
subjects        : 86
per class       : {'BL1': 1668, 'PA1': 1671, 'PA2': 1686, 'PA3': 1700, 'PA4': 1701}
per subject     : min=71 median=100 max=100
subjects < 90   : {'083013_w_47': 77, '083114_w_55': 71, '091814_m_37': 78}
fs_hz           : 512.0328
cache           : 86 files, 453 MiB


## Step 2 — feature extraction

232 features per trial: 61 EDA, 51 ECG and 40 per EMG channel (trapezius, corrugator, zygomaticus;
50/100 Hz notch applied first). Baseline-similarity features (correlation, coherence, mutual information
with the subject's mean BL1 signal) are computed twice: with the full BL1 template, and with a
leave-one-out template that excludes the trial itself (`loo__` columns), so no BL1 trial is compared with itself.

In [3]:
from scipy.signal import (butter, coherence, filtfilt, find_peaks, iirnotch, peak_widths,
                          sosfiltfilt, stft, welch)
from scipy.stats import kurtosis, skew
from sklearn.metrics import mutual_info_score

SIM_SPEC = {   # similarity band and window lengths per modality
    "gsr": dict(band=(0.05, 1.5), coh_nperseg=1024, welch_nperseg=2048, mi=True),
    "ecg": dict(band=(0.05, 1.5), coh_nperseg=1024, welch_nperseg=1024, mi=False),
    "emg": dict(band=(10.0, 150.0), coh_nperseg=512, welch_nperseg=1024, mi=True),
}


# ---------------- generic helpers ----------------
def safe_moment(fn, x, min_n, **kw):
    if len(x) < min_n or float(np.std(x)) < 1e-12:
        return 0.0
    v = float(fn(x, **kw))
    return v if np.isfinite(v) else 0.0


def basic_stats(prefix, x, unbiased=False):
    q05, q25, q75, q95 = np.percentile(x, [5, 25, 75, 95])
    dx = np.diff(x)
    if unbiased:
        sk = safe_moment(skew, x, 3, bias=False)
        ku = safe_moment(kurtosis, x, 4, fisher=True, bias=False)
    else:
        sk, ku = float(skew(x)), float(kurtosis(x))
    return {f"{prefix}_{k}": v for k, v in [
        ("mean", float(np.mean(x))), ("std", float(np.std(x))), ("min", float(np.min(x))),
        ("max", float(np.max(x))), ("median", float(np.median(x))), ("iqr", float(q75 - q25)),
        ("skew", sk), ("kurtosis", ku), ("p05", float(q05)), ("p25", float(q25)), ("p75", float(q75)),
        ("p95", float(q95)), ("ptp", float(np.ptp(x))),
        ("diff_abs_mean", float(np.mean(np.abs(dx)))), ("diff_std", float(np.std(dx))),
    ]}


def hjorth(x):
    d1 = np.diff(x)
    var0, var1, var2 = float(np.var(x)), float(np.var(d1)), float(np.var(np.diff(d1)))
    mobility = float(np.sqrt(var1 / var0)) if var0 > 0 else 0.0
    complexity = float(np.sqrt(var2 / var1) / mobility) if var1 > 0 and mobility > 0 else 0.0
    return var0, mobility, complexity


def psd(x, fs, nperseg):
    n = len(x)
    f, pxx = welch(x, fs=fs, nperseg=min(nperseg, n), noverlap=min(nperseg // 2, min(nperseg, n) // 2),
                   detrend="constant", scaling="density")
    return f, np.maximum(pxx, 0.0)


def band_power(f, pxx, lo, hi):
    m = (f >= lo) & (f < hi)
    return float(np.sum(pxx[m]) * (f[1] - f[0])) if m.any() else 0.0     # sum of bins * df


def spectral_shape(f, pxx, lo, hi, keys):
    m = (f >= lo) & (f <= hi)
    ff, pp = f[m], pxx[m]
    p_sum = float(pp.sum())
    if p_sum <= 0 or len(ff) < 2:
        return {k: 0.0 for k in keys}
    p = pp / p_sum
    p = p[p > 0]
    cum = np.cumsum(pp)
    out = {
        "centroid": float(np.sum(ff * pp) / p_sum),
        "entropy": float(-np.sum(p * np.log(p))),
        "peak_freq": float(ff[np.argmax(pp)]),
        "median_freq": float(ff[np.searchsorted(cum, 0.50 * cum[-1])]),
        "edge95": float(ff[np.searchsorted(cum, 0.95 * cum[-1])]),
        "slope": float(np.polyfit(np.log10(ff), np.log10(pp + 1e-20), 1)[0]),
    }
    return {k: out[k] for k in keys}


def rel(parts, total):
    return [p / total if total > 0 else 0.0 for p in parts]


def binned_mutual_info(x, y, bins=16):
    qx = np.unique(np.quantile(x, np.linspace(0.0, 1.0, bins + 1)))
    qy = np.unique(np.quantile(y, np.linspace(0.0, 1.0, bins + 1)))
    if len(qx) < 2 or len(qy) < 2:
        return 0.0
    return float(mutual_info_score(np.digitize(x, qx[1:-1]), np.digitize(y, qy[1:-1])))


def similarity_features(x, base, fs, spec):
    """Correlation, magnitude-squared coherence (mean/median/PSD-weighted) and binned MI between a trial and a template."""
    n = len(x)
    c = float(np.corrcoef(x, base)[0, 1]) if x.std() > 0 and base.std() > 0 else 0.0
    out = {"corr": c if np.isfinite(c) else 0.0}
    f, pxx = welch(x, fs=fs, nperseg=min(spec["welch_nperseg"], n), noverlap=min(spec["welch_nperseg"] // 2, n // 2))
    np_c = min(spec["coh_nperseg"], n)
    with np.errstate(invalid="ignore", divide="ignore"):          # 0/0 where both PSDs vanish
        f2, coh = coherence(x, base, fs=fs, nperseg=np_c, noverlap=min(spec["coh_nperseg"] // 2, np_c // 2),
                            detrend="constant")
    valid = (f2 >= spec["band"][0]) & (f2 <= spec["band"][1]) & np.isfinite(coh)
    if valid.any():
        v, w = coh[valid], np.interp(f2[valid], f, pxx)
        out["coh_mean"], out["coh_median"] = float(np.mean(v)), float(np.median(v))
        out["coh_weighted"] = float(np.average(v, weights=w)) if w.sum() > 0 else 0.0
    else:
        out["coh_mean"] = out["coh_median"] = out["coh_weighted"] = 0.0
    if spec["mi"]:
        out["mi"] = binned_mutual_info(x, base)
    return out


# ---------------- EDA / GSR ----------------
def eda_features(x, fs):
    n = len(x)
    t = np.arange(n) / fs
    feats = basic_stats("raw", x)
    feats["raw_slope"] = float(np.polyfit(t, x, 1)[0])

    tonic = sosfiltfilt(butter(2, 0.05, fs=fs, btype="low", output="sos"), x)
    phasic = x - tonic
    feats["tonic_std"], feats["tonic_ptp"] = float(np.std(tonic)), float(np.ptp(tonic))
    feats["tonic_slope"] = float(np.polyfit(t, tonic, 1)[0])
    feats.update(basic_stats("phasic", phasic))
    feats["hjorth_activity"], feats["hjorth_mobility"], feats["hjorth_complexity"] = hjorth(x)

    # SCR-like peaks: relative prominence on the z-scored signal, not a hard amplitude threshold
    z = (x - x.mean()) / x.std() if x.std() > 0 else np.zeros_like(x)
    peaks, props = find_peaks(z, prominence=0.10, distance=max(1, int(0.5 * fs)))
    feats["scr_peaks_n"] = float(len(peaks))
    feats["scr_peaks_per_sec"] = float(len(peaks) / (n / fs))
    if len(peaks):
        prom = props["prominences"]
        widths = peak_widths(z, peaks, rel_height=0.5)[0] / fs
        feats.update(scr_prom_mean=float(prom.mean()), scr_prom_max=float(prom.max()),
                     scr_prom_sum=float(prom.sum()), scr_width_mean_s=float(widths.mean()))
    else:
        feats.update(scr_prom_mean=0.0, scr_prom_max=0.0, scr_prom_sum=0.0, scr_width_mean_s=0.0)
    feats["scr_positive_area"] = float(np.sum(np.clip(phasic, 0.0, None)) / fs)

    f, pxx = psd(phasic, fs, 2048)                                  # 0.25 Hz resolution
    bp = [band_power(f, pxx, lo, hi) for lo, hi in [(0.05, 0.5), (0.5, 1.5), (1.5, 5.0)]]
    total = sum(bp)
    feats.update(spec_bp_005_05=bp[0], spec_bp_05_15=bp[1], spec_bp_15_50=bp[2])
    feats.update(zip(["spec_rel_bp_005_05", "spec_rel_bp_05_15", "spec_rel_bp_15_50"], rel(bp, total)))
    feats["spec_total_power_005_5"] = total
    feats.update({f"spec_{k}": v for k, v in
                  spectral_shape(f, pxx, 0.05, 5.0, ["centroid", "entropy", "peak_freq", "edge95", "slope"]).items()})
    return feats


# ---------------- ECG ----------------
def detect_r_peaks(x, fs):
    """Envelope QRS detector: 5-20 Hz band-pass, 80 ms smoothing, 90th-percentile threshold, 0.3 s refractory.
    The threshold multiplier is raised only while the detector over-counts (> 180 bpm)."""
    if len(x) < int(fs):
        return np.array([], dtype=int), 0.0
    env = np.abs(sosfiltfilt(butter(3, (5.0, 20.0), btype="bandpass", fs=fs, output="sos"), x))
    win = max(1, int(0.08 * fs))
    env = np.convolve(env, np.ones(win) / win, mode="same")
    base_thr = float(np.quantile(env, 0.90))
    if not np.isfinite(base_thr) or base_thr <= 0:
        base_thr = 0.5 * float(np.max(env)) if np.max(env) > 0 else 0.0
    if base_thr <= 0:
        return np.array([], dtype=int), 0.0
    min_dist = max(1, int(0.30 * fs))
    peaks, thr_used = np.array([], dtype=int), 0.0
    for mult in (1.0, 1.15, 1.30, 1.50, 1.75):
        thr = base_thr * mult
        cand, _ = find_peaks(env, distance=min_dist, height=thr)
        if len(cand) < 2:
            if len(peaks) == 0:
                peaks, thr_used = cand, thr
            break
        peaks, thr_used = cand, thr
        if 60.0 * fs / float(np.mean(np.diff(cand))) <= 180.0:
            break
    if len(peaks) < 2:                                              # severe under-detection fallback
        thr = float(np.quantile(env, 0.80))
        cand, _ = find_peaks(env, distance=min_dist, height=thr)
        if len(cand) > len(peaks):
            peaks, thr_used = cand, thr
    return peaks.astype(int), float(thr_used)


def hrv_features(peaks, fs):
    """Short-window time-domain HRV; 5.5 s segments do not support LF/HF estimates."""
    keys = ["n_peaks", "hr_mean_bpm", "hr_std_bpm", "rr_mean_s", "rr_median_s", "rr_std_s", "rr_cv",
            "rr_min_s", "rr_max_s", "rr_range_s", "rmssd_s", "pnn50", "rr_slope"]
    out = dict.fromkeys(keys, 0.0)
    out["n_peaks"] = float(len(peaks))
    rr = np.diff(peaks) / fs if len(peaks) >= 2 else np.array([])
    rr = rr[np.isfinite(rr) & (rr > 0)]
    if len(rr) == 0:
        return out
    rr_mean, rr_std = float(np.mean(rr)), float(np.std(rr))
    out.update(hr_mean_bpm=60.0 / rr_mean, hr_std_bpm=float(np.std(60.0 / rr)), rr_mean_s=rr_mean,
               rr_median_s=float(np.median(rr)), rr_std_s=rr_std, rr_cv=rr_std / rr_mean,
               rr_min_s=float(np.min(rr)), rr_max_s=float(np.max(rr)), rr_range_s=float(np.ptp(rr)))
    if len(rr) >= 2:
        drr = np.diff(rr)
        out["rmssd_s"] = float(np.sqrt(np.mean(drr ** 2)))
        out["pnn50"] = float(np.mean(np.abs(drr) > 0.05))
    if len(rr) >= 3 and float(np.ptp(rr)) > 0:
        out["rr_slope"] = float(np.polyfit(np.arange(len(rr)), rr, 1)[0])
    return out


def ecg_features(x, fs):
    n = len(x)
    feats = basic_stats("raw", x, unbiased=True)
    feats["raw_rms"] = float(np.sqrt(np.mean(x ** 2)))
    feats["raw_slope"] = float(np.polyfit(np.arange(n) / fs, x, 1)[0])
    feats["abs_mean"] = float(np.mean(np.abs(x)))
    feats["zero_crossing_rate"] = float(np.count_nonzero(np.diff(np.signbit(x - x.mean()).astype(np.int8))) / max(1, n - 1))
    _, feats["hjorth_mobility"], feats["hjorth_complexity"] = hjorth(x)

    f, pxx = psd(x, fs, 1024)
    bp = [band_power(f, pxx, lo, hi) for lo, hi in [(0.5, 5.0), (5.0, 15.0), (15.0, 40.0)]]
    total = sum(bp)
    feats.update(spec_bp_05_5=bp[0], spec_bp_5_15=bp[1], spec_bp_15_40=bp[2], spec_total_05_40=total)
    feats.update(zip(["spec_rel_bp_05_5", "spec_rel_bp_5_15", "spec_rel_bp_15_40"], rel(bp, total)))
    feats.update({f"spec_{k}": v for k, v in
                  spectral_shape(f, pxx, 0.5, 40.0, ["centroid", "entropy", "peak_freq", "edge95", "slope"]).items()})

    peaks, feats["det_thr"] = detect_r_peaks(x, fs)
    feats.update(hrv_features(peaks, fs))
    return feats


# ---------------- EMG ----------------
def notch_filter(block, fs):
    """Suppress 50 Hz mains and its first harmonic (German recordings); works on (n, L) or (L,)."""
    for f0 in (50.0, 100.0):
        if f0 < fs / 2:
            b, a = iirnotch(f0, 30.0, fs=fs)
            block = filtfilt(b, a, block, axis=-1)
    return block


def burst_stats(x, fs, n_windows=20):
    win = max(1, len(x) // n_windows)
    win_rms = np.array([np.sqrt(np.mean(x[j:j + win] ** 2)) for j in range(0, len(x) - win + 1, win)])
    mean_rms, std_rms = float(win_rms.mean()), float(win_rms.std())
    above = win_rms > mean_rms + std_rms
    n_bursts = int(np.count_nonzero(np.diff(np.r_[False, above, False].astype(np.int8)) == 1))
    return {
        "win_rms_cv": std_rms / mean_rms if mean_rms > 0 else 0.0,
        "win_rms_max_ratio": float(win_rms.max()) / mean_rms if mean_rms > 0 else 0.0,
        "burst_rate": n_bursts / (len(x) / fs),
        "burst_duty": float(above.mean()),
    }


def spectral_flux(x, fs):
    f, _, Z = stft(x, fs=fs, nperseg=256, noverlap=128, detrend=False, scaling="spectrum", boundary=None, padded=False)
    P = np.abs(Z[(f >= 10.0) & (f <= 150.0)]) ** 2
    if P.shape[1] < 2:
        return 0.0
    P = P / (P.sum(axis=0, keepdims=True) + 1e-20)
    return float(np.mean(np.abs(np.diff(P, axis=1))))


def emg_features(x, fs):
    """x must already be notch-filtered."""
    n = len(x)
    duration = n / fs
    abs_x, dx = np.abs(x), np.diff(x)
    q25, q75 = np.percentile(x, [25, 75])
    rms = float(np.sqrt(np.mean(x ** 2)))
    feats = {
        "mean": float(np.mean(x)), "std": float(np.std(x)), "median": float(np.median(x)), "iqr": float(q75 - q25),
        "skew": safe_moment(skew, x, 3, bias=False), "kurtosis": safe_moment(kurtosis, x, 4, fisher=True, bias=False),
        "rms": rms, "log_rms": float(np.log(rms + 1e-12)), "mav": float(np.mean(abs_x)), "ptp": float(np.ptp(x)),
        "p95_abs": float(np.percentile(abs_x, 95)), "waveform_length": float(np.sum(np.abs(dx))),
        "zero_crossing_rate": float(np.count_nonzero(np.diff(np.signbit(x - x.mean()).astype(np.int8))) / duration),
        "slope_sign_changes": float(np.count_nonzero(np.diff(np.signbit(dx).astype(np.int8))) / duration),
    }
    _, feats["hjorth_mobility"], feats["hjorth_complexity"] = hjorth(x)

    f, pxx = psd(x, fs, 1024)
    bands = [(10.0, 30.0), (30.0, 60.0), (60.0, 100.0), (100.0, 150.0)]
    bp = [band_power(f, pxx, lo, hi) for lo, hi in bands]
    total = band_power(f, pxx, 10.0, 150.0)
    feats.update(spec_bp_10_30=bp[0], spec_bp_30_60=bp[1], spec_bp_60_100=bp[2], spec_bp_100_150=bp[3],
                 spec_total_10_150=total)
    feats.update(zip(["spec_rel_bp_10_30", "spec_rel_bp_30_60", "spec_rel_bp_60_100", "spec_rel_bp_100_150"],
                     rel(bp, total)))
    feats.update({f"spec_{k}": v for k, v in
                  spectral_shape(f, pxx, 10.0, 150.0, ["centroid", "entropy", "median_freq", "edge95", "slope"]).items()})
    feats.update(burst_stats(x, fs))
    feats["spec_flux"] = spectral_flux(x, fs)
    return feats


# ---------------- one subject ----------------
def extract_subject(cache_file, fs_per_trial, class_ids):
    """All features of one subject's cached trials, with full and leave-one-out BL1 templates."""
    arr = np.load(CACHE_DIR / cache_file).astype(np.float64)        # (n_trials, 5, SEG_LEN)
    assert len(arr) == len(class_ids) == len(fs_per_trial)
    bl1 = np.flatnonzero(class_ids == 0)
    assert len(bl1) >= 2, f"{cache_file}: fewer than 2 BL1 trials"
    for ci in EMG_CHANNELS.values():
        arr[:, ci, :] = notch_filter(arr[:, ci, :], float(fs_per_trial[0]))
    bl1_sum = arr[bl1].sum(axis=0)
    full_template = bl1_sum / len(bl1)

    def sim_block(prefix, x, full_t, loo_t, fs, spec):
        full = similarity_features(x, full_t, fs, spec)
        loo = similarity_features(x, loo_t, fs, spec)
        row = {f"{prefix}_sim_{k}": v for k, v in full.items()}
        row.update({f"{LOO_PREFIX}{prefix}_sim_{k}": v for k, v in loo.items()})
        return row

    rows = []
    for i, (x, fs) in enumerate(zip(arr, fs_per_trial)):
        fs = float(fs)
        loo_template = (bl1_sum - x) / (len(bl1) - 1) if class_ids[i] == 0 else full_template
        row = {f"gsr_{k}": v for k, v in eda_features(x[0], fs).items()}
        row.update(sim_block("gsr", x[0], full_template[0], loo_template[0], fs, SIM_SPEC["gsr"]))
        row.update({f"ecg_{k}": v for k, v in ecg_features(x[1], fs).items()})
        row.update(sim_block("ecg", x[1], full_template[1], loo_template[1], fs, SIM_SPEC["ecg"]))
        for name, ci in EMG_CHANNELS.items():
            row.update({f"emg_{name}_{k}": v for k, v in emg_features(x[ci], fs).items()})
            row.update(sim_block(f"emg_{name}", x[ci], full_template[ci], loo_template[ci], fs, SIM_SPEC["emg"]))
        rows.append(row)
    return rows

In [4]:
meta = pd.read_parquet(CACHE_DIR / "meta.parquet").sort_values("sample_id").reset_index(drop=True)
groups = []
for _, g in meta.groupby("subject_id", sort=True):
    assert np.array_equal(g["cache_row"].to_numpy(), np.arange(len(g)))   # cache order == sample order
    assert np.allclose(g["fs_hz"], g["fs_hz"].iloc[0])
    groups.append(g)

t0 = time.perf_counter()
results = Parallel(n_jobs=N_JOBS)(
    delayed(extract_subject)(g["cache_file"].iloc[0], g["fs_hz"].to_numpy(), g["class_id"].to_numpy())
    for g in groups
)
print(f"extracted features for {len(meta)} trials in {time.perf_counter() - t0:.1f}s")

features = pd.concat(
    [g[ID_COLS].reset_index(drop=True).join(pd.DataFrame(rows)) for g, rows in zip(groups, results)],
    ignore_index=True,
).sort_values("sample_id").reset_index(drop=True)

feature_cols = [c for c in features.columns if c.startswith(("gsr_", "ecg_", "emg_"))]
loo_cols = [c for c in features.columns if c.startswith(LOO_PREFIX)]
values = features[feature_cols + loo_cols].to_numpy(dtype=np.float64)

# ---- validation ----
assert len(features) == len(meta) and features["sample_id"].equals(meta["sample_id"])
assert np.isfinite(values).all(), "NaN or infinity in features"
counts = {m: sum(c.startswith(p) for c in feature_cols) for m, p in [("eda", "gsr_"), ("ecg", "ecg_"), ("emg", "emg_")]}
assert counts == {"eda": 61, "ecg": 51, "emg": 120}, counts
assert len(loo_cols) == 24, len(loo_cols)
pain = features["class_id"].to_numpy() != 0
for c in loo_cols:                                                # LOO only changes BL1 rows
    assert np.array_equal(features.loc[pain, c].to_numpy(), features.loc[pain, c[len(LOO_PREFIX):]].to_numpy()), c

std = features[feature_cols].std()
zero_frac = (features[feature_cols] == 0).mean().sort_values(ascending=False)
print("\nfeatures          :", len(feature_cols), counts, "+", len(loo_cols), "leave-one-out similarity columns")
print("zero-variance     :", std[std <= 0].index.tolist() or "none")
print("most-zero features:")
print(zero_frac.head(6).round(3).to_string())
hr = features["ecg_hr_mean_bpm"]
print("\nECG R-peaks/trial : min=%d median=%d max=%d   HR bpm: min=%.0f median=%.0f max=%.0f"
      % (features["ecg_n_peaks"].min(), features["ecg_n_peaks"].median(), features["ecg_n_peaks"].max(),
         hr.min(), hr.median(), hr.max()))

features.to_parquet(FEATURES_DIR / "features_raw_partB.parquet", index=False)
write_json(FEATURES_DIR / "features_raw_manifest.json", {
    "n_samples": int(len(features)),
    "n_subjects": int(features["subject_id"].nunique()),
    "n_features": len(feature_cols),
    "features_per_modality": counts,
    "feature_columns": feature_cols,
    "loo_columns": loo_cols,
    "eda": {"tonic": "2nd-order Butterworth low-pass 0.05 Hz (zero-phase)",
            "scr_peaks": "prominence 0.10 on z-scored GSR, min distance 0.5 s",
            "welch": {"nperseg": 2048, "noverlap": 1024, "bands_hz": [[0.05, 0.5], [0.5, 1.5], [1.5, 5.0]]}},
    "ecg": {"r_peak_detector": {"bandpass_hz": [5.0, 20.0], "order": 3, "envelope_smoothing_s": 0.08,
                                "base_threshold_quantile": 0.90, "min_distance_s": 0.30, "overdetection_guard_bpm": 180},
            "welch": {"nperseg": 1024, "noverlap": 512, "bands_hz": [[0.5, 5.0], [5.0, 15.0], [15.0, 40.0]]},
            "hrv_note": "5.5 s segments: time-domain HRV only, no LF/HF"},
    "emg": {"notch_hz": [50.0, 100.0], "notch_q": 30.0,
            "welch": {"nperseg": 1024, "noverlap": 512, "bands_hz": [[10, 30], [30, 60], [60, 100], [100, 150]]},
            "stationarity": {"n_rms_windows": 20, "burst_threshold": "mean + std of window RMS",
                             "spectral_flux_stft": {"nperseg": 256, "noverlap": 128, "band_hz": [10, 150]}}},
    "similarity": {k: {kk: vv for kk, vv in v.items()} for k, v in SIM_SPEC.items()},
    "similarity_template": "mean of the subject's retained BL1 trials; loo__ columns exclude the trial itself for BL1 rows",
    "band_power_integration": "sum(pxx_bins) * df",
})
print("\nwrote:", FEATURES_DIR / "features_raw_partB.parquet", features.shape)

extracted features for 8426 trials in 15.8s



features          : 232 {'eda': 61, 'ecg': 51, 'emg': 120} + 24 leave-one-out similarity columns
zero-variance     : none
most-zero features:
ecg_pnn50                0.549
gsr_scr_peaks_per_sec    0.274
gsr_scr_peaks_n          0.274
gsr_scr_prom_mean        0.274
gsr_scr_prom_sum         0.274
gsr_scr_width_mean_s     0.274

ECG R-peaks/trial : min=4 median=6 max=10   HR bpm: min=36 median=68 max=111

wrote: D:\HGB\Final_pipeline\outputs\features\features_raw_partB.parquet (8426, 262)


## Step 3 — feature hygiene

Constant features are dropped, then a greedy pass in column order removes any feature with |r| > 0.95
to an already kept feature. The correlation basis is within-subject z-scored data, which is what the models receive.
Two model tables are written with identical columns: full-template similarity and leave-one-out similarity.

In [5]:
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score


def zscore_within_subject(X, subject, eps=1e-12):
    """Centre and scale each feature with the subject's own (unlabelled) trials; features flat within a subject become 0."""
    Z = np.array(X, dtype=np.float64, copy=True)
    flat = np.zeros(Z.shape[1], dtype=int)
    for sid in np.unique(subject):
        pos = subject == sid
        mu, sd = Z[pos].mean(axis=0), Z[pos].std(axis=0)
        flat += sd <= eps
        Z[pos] = (Z[pos] - mu) / np.where(sd <= eps, 1.0, sd)
    return Z, flat


def core_metrics(y_true, y_pred, labels):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "kappa_quadratic": float(cohen_kappa_score(y_true, y_pred, labels=labels, weights="quadratic")),
        "kappa_unweighted": float(cohen_kappa_score(y_true, y_pred, labels=labels)),
        "mae": float(np.mean(np.abs(y_true - y_pred))),
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "within_one_level": float(np.mean(np.abs(y_true - y_pred) <= 1)),
    }


CORR_THRESHOLD = 0.95
modality_of = lambda c: "eda" if c.startswith("gsr_") else c.split("_")[0]

raw = pd.read_parquet(FEATURES_DIR / "features_raw_partB.parquet").sort_values("sample_id").reset_index(drop=True)
feature_cols = [c for c in raw.columns if c.startswith(("gsr_", "ecg_", "emg_"))]
subject = raw["subject_id"].to_numpy()

# ---- constants ----
std = raw[feature_cols].std(ddof=0)
constant = std.index[std <= 1e-12].tolist()
active = [c for c in feature_cols if c not in constant]

# ---- greedy de-duplication on within-subject z-scores ----
Z, flat_counts = zscore_within_subject(raw[active].to_numpy(), subject)
corr = np.nan_to_num(np.corrcoef(Z, rowvar=False), nan=0.0)
np.fill_diagonal(corr, 0.0)

kept, log = [], []
for j, name in enumerate(active):
    r = np.abs(corr[j, kept]) if kept else np.array([])
    if len(r) and r.max() > CORR_THRESHOLD:
        w = int(np.argmax(r))
        log.append({"feature": name, "modality": modality_of(name), "status": "dropped_duplicate",
                    "duplicate_of": active[kept[w]], "abs_r": float(r[w])})
    else:
        kept.append(j)
        log.append({"feature": name, "modality": modality_of(name), "status": "kept", "duplicate_of": "", "abs_r": np.nan})
kept_cols = [active[j] for j in kept]
dedupe = pd.DataFrame(log)
max_r_kept = float(np.abs(corr[np.ix_(kept, kept)]).max())
assert max_r_kept <= CORR_THRESHOLD

# ---- model tables: full-template and leave-one-out similarity ----
model_full = raw[ID_COLS + kept_cols].copy()
model_loo = model_full.copy()
sim_kept = [c for c in kept_cols if "_sim_" in c]
for c in sim_kept:
    model_loo[c] = raw[LOO_PREFIX + c].to_numpy()

Z_loo, _ = zscore_within_subject(model_loo[kept_cols].to_numpy(), subject)
C = np.nan_to_num(np.corrcoef(Z_loo, rowvar=False), nan=0.0)
np.fill_diagonal(C, 0.0)
pairs_over = int((np.abs(C) > CORR_THRESHOLD).sum() // 2)

bl1 = raw["class_id"].to_numpy() == 0
before_after = pd.DataFrame({
    "feature": sim_kept,
    "bl1_mean_full": [float(model_full.loc[bl1, c].mean()) for c in sim_kept],
    "bl1_mean_loo": [float(model_loo.loc[bl1, c].mean()) for c in sim_kept],
})
before_after["delta"] = before_after["bl1_mean_loo"] - before_after["bl1_mean_full"]

# ---- report ----
print("input features    :", len(feature_cols))
print("dropped constants :", len(constant), constant)
print("dropped duplicates:", int((dedupe["status"] == "dropped_duplicate").sum()))
print("kept features     :", len(kept_cols), dedupe.loc[dedupe["status"] == "kept", "modality"].value_counts().to_dict())
print("max |r| among kept: %.4f (full template)   pairs > %.2f on LOO basis: %d" % (max_r_kept, CORR_THRESHOLD, pairs_over))
flat_in_subject = {active[j]: int(n) for j, n in enumerate(flat_counts) if n > 0}
print("flat within some subjects:", flat_in_subject or "none")
print("\ndropped duplicates (feature -> kept partner):")
print(dedupe.loc[dedupe["status"] == "dropped_duplicate", ["modality", "feature", "duplicate_of", "abs_r"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nBL1 similarity, full template -> leave-one-out (mean over BL1 trials):")
print(before_after.sort_values("delta").to_string(index=False, float_format=lambda v: f"{v: .4f}"))

model_full.to_parquet(FEATURES_DIR / "features_model_partB.parquet", index=False)
model_loo.to_parquet(FEATURES_DIR / "features_model_partB_loo.parquet", index=False)
dedupe.to_parquet(FEATURES_DIR / "feature_dedupe_log.parquet", index=False)
write_json(FEATURES_DIR / "features_model_manifest.json", {
    "n_samples": int(len(model_full)),
    "n_subjects": int(model_full["subject_id"].nunique()),
    "class_counts": {CLASS_NAMES[int(k)]: int(v) for k, v in model_full["class_id"].value_counts().sort_index().items()},
    "n_features_in": len(feature_cols),
    "constant_features": constant,
    "n_duplicate_dropped": int((dedupe["status"] == "dropped_duplicate").sum()),
    "n_features_out": len(kept_cols),
    "feature_columns": kept_cols,
    "similarity_columns": sim_kept,
    "dropped_features": dedupe.loc[dedupe["status"] == "dropped_duplicate", ["feature", "duplicate_of", "abs_r"]].to_dict("records"),
    "dedupe": {"basis": "within-subject z-scored", "threshold": CORR_THRESHOLD,
               "rule": "greedy in column order; drop when |r| > threshold with an already kept feature",
               "max_abs_r_among_kept": max_r_kept, "pairs_over_threshold_on_loo_basis": pairs_over},
    "features_flat_in_some_subject": flat_in_subject,
    "tables": {"features_model_partB.parquet": "full BL1 template similarity",
               "features_model_partB_loo.parquet": "leave-one-out BL1 template similarity (BL1 rows differ only)"},
    "bl1_similarity_full_vs_loo": before_after.to_dict("records"),
})
print("\nwrote:", FEATURES_DIR / "features_model_partB.parquet", "and", FEATURES_DIR / "features_model_partB_loo.parquet")

input features    : 232
dropped constants : 0 []
dropped duplicates: 57
kept features     : 175 {'emg': 97, 'ecg': 43, 'eda': 35}
max |r| among kept: 0.9495 (full template)   pairs > 0.95 on LOO basis: 0
flat within some subjects: {'gsr_spec_peak_freq': 26, 'ecg_spec_peak_freq': 2, 'ecg_pnn50': 2}

dropped duplicates (feature -> kept partner):
modality                           feature                   duplicate_of  abs_r
     eda                       gsr_raw_min                   gsr_raw_mean  0.966
     eda                       gsr_raw_max                   gsr_raw_mean  0.957
     eda                    gsr_raw_median                   gsr_raw_mean  0.994
     eda                       gsr_raw_iqr                    gsr_raw_std  0.956
     eda                       gsr_raw_p05                   gsr_raw_mean  0.970
     eda                       gsr_raw_p25                   gsr_raw_mean  0.983
     eda                       gsr_raw_p75                   gsr_raw_mean  0.990
     e

## Step 4 — LOSO classification

Both models are trained on identical folds (one subject held out per fold) after within-subject z-scoring,
which uses only the subject's own unlabelled trials. Six configurations:

| run | table | features | classes | purpose |
|---|---|---:|---:|---|
| `full_template_5class` | full template | 175 | 5 | contaminated: a BL1 trial contributes to its own template |
| `loo_5class` | leave-one-out | 175 | 5 | self-inclusion removed; still exposed to the BL1/PA1 facial-EMG copy (Steps 6 and 9) |
| `loo_5class_169` | leave-one-out | 169 | 5 | six facial similarity columns removed: leakage-controlled 5-class estimate |
| `loo_4class_pa1ex` | leave-one-out | 175 | 4 | PA1 excluded: no twin pairs left, so no leakage |
| `loo_5class_nofacial` | leave-one-out | ECG, EDA, trapezius | 5 | no corrugator or zygomaticus feature at all |
| `loo_4class_nofacial` | leave-one-out | ECG, EDA, trapezius | 4 | PA1 excluded, no facial EMG |

Step 9 shows that the released BL1 facial EMG are copies of PA1 recordings, so the facial channels carry PA1 signals in
the BL1 class. The `nofacial` runs use only channels that are intact in every class.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

model_manifest = json.loads((FEATURES_DIR / "features_model_manifest.json").read_text(encoding="utf-8"))
kept_cols = model_manifest["feature_columns"]
model_full = pd.read_parquet(FEATURES_DIR / "features_model_partB.parquet")
model_loo = pd.read_parquet(FEATURES_DIR / "features_model_partB_loo.parquet")

# Columns that encode the PA1/BL1 facial duplication (see Steps 6 and 9), and all facial-EMG columns.
FACIAL_SIM_COLS = [f"emg_{ch}_sim_{k}" for ch in ("corrugator", "zygomaticus") for k in ("corr", "coh_mean", "mi")]
assert set(FACIAL_SIM_COLS) <= set(kept_cols), set(FACIAL_SIM_COLS) - set(kept_cols)
FACIAL_COLS = [c for c in kept_cols if c.startswith(("emg_corrugator_", "emg_zygomaticus_"))]
NOFACIAL_COLS = [c for c in kept_cols if c not in FACIAL_COLS]
FOUR_CLASS = {"label_map": {0: 0, 2: 1, 3: 2, 4: 3}, "names": {0: "BL1", 1: "PA2", 2: "PA3", 3: "PA4"}}


def fit_predict(model, X_train, y_train, X_test, n_classes):
    model.fit(X_train, y_train)
    proba = np.zeros((len(X_test), n_classes))
    raw = model.predict_proba(X_test)
    for j, cls in enumerate(model.classes_):
        proba[:, int(cls)] = raw[:, j]
    return proba


def prepare_run(table, feature_cols, label_map=None):
    """Select and relabel the classes of a run, then z-score the features within subject."""
    df = table.sort_values("sample_id").reset_index(drop=True)
    if label_map is not None:
        df = df[df["class_id"].isin(label_map)].reset_index(drop=True)
    y = df["class_id"].map(label_map or {k: k for k in LABELS}).to_numpy(dtype=int)
    groups = df["subject_id"].to_numpy(dtype=int)
    X, _ = zscore_within_subject(df[feature_cols].to_numpy(), groups)
    assert np.isfinite(X).all()
    return df, X, y, sorted(set(y.tolist())), groups


def run_loso(name, table, feature_cols, label_map=None, note=""):
    """LOSO with XGBoost and HGB on identical folds. Saves predictions, fold stats, XGB gain importance and a manifest."""
    df, X, y, labels, groups = prepare_run(table, feature_cols, label_map)
    sample_ids = df["sample_id"].to_numpy(dtype=int)
    subjects = np.unique(groups)

    print("=" * 78)
    print(f"{name}: {len(df)} trials, {len(subjects)} subjects, {len(feature_cols)} features, {len(labels)} classes")
    blocks, fold_rows = [], []
    gain = np.zeros(len(feature_cols))
    t_start = time.perf_counter()
    for fold, sid in enumerate(subjects):
        test = groups == sid
        train = ~test
        t0 = time.perf_counter()
        xgb = XGBClassifier(**(XGB_PARAMS if len(labels) > 2 else XGB_BINARY_PARAMS))
        proba = {"xgb": fit_predict(xgb, X[train], y[train], X[test], len(labels))}
        t_xgb = time.perf_counter() - t0
        gain += xgb.feature_importances_                      # per-fold gain shares sum to 1
        t0 = time.perf_counter()
        proba["hgb"] = fit_predict(HistGradientBoostingClassifier(**HGB_PARAMS), X[train], y[train], X[test], len(labels))
        t_hgb = time.perf_counter() - t0

        row = {"fold": fold + 1, "subject_id": int(sid), "n_train": int(train.sum()), "n_test": int(test.sum()),
               "xgb_seconds": t_xgb, "hgb_seconds": t_hgb}
        for m in MODELS:
            pred = proba[m].argmax(axis=1)
            block = pd.DataFrame({"model": m, "sample_id": sample_ids[test], "subject_id": groups[test],
                                  "true_class": y[test], "pred_class": pred})
            for j in labels:
                block[f"prob_{j}"] = proba[m][:, j]
            blocks.append(block)
            row[f"{m}_acc"] = float(accuracy_score(y[test], pred))
        fold_rows.append(row)
        if (fold + 1) % 10 == 0 or fold + 1 == len(subjects):
            elapsed = time.perf_counter() - t_start
            print(f"  fold {fold + 1:2d}/{len(subjects)}  {elapsed:6.0f}s  eta {elapsed / (fold + 1) * (len(subjects) - fold - 1) / 60:4.1f} min")

    preds = pd.concat(blocks, ignore_index=True)
    folds = pd.DataFrame(fold_rows)
    prob_cols = [f"prob_{j}" for j in labels]
    assert folds["n_test"].sum() == len(df) and len(preds) == 2 * len(df)
    assert not preds.duplicated(["model", "sample_id"]).any()
    assert np.allclose(preds[prob_cols].sum(axis=1), 1.0, atol=1e-6)

    summary = {}
    for m in MODELS:
        b = preds[preds["model"] == m]
        summary[m] = core_metrics(b["true_class"].to_numpy(), b["pred_class"].to_numpy(), labels)
        print(f"  {MODEL_NAMES[m]:<8s} acc {summary[m]['accuracy']:.4f}  macro-F1 {summary[m]['macro_f1']:.4f}  "
              f"kappa_q {summary[m]['kappa_quadratic']:.4f}  MAE {summary[m]['mae']:.3f}  "
              f"fold acc median {folds[f'{m}_acc'].median():.3f}")
    gain_df = (pd.DataFrame({"feature": feature_cols, "mean_gain": gain / len(subjects)})
               .sort_values("mean_gain", ascending=False).reset_index(drop=True))
    gain_df["rank"] = np.arange(1, len(gain_df) + 1)
    print("  top-5 XGB gain:", ", ".join(f"{f} {g:.3f}" for f, g in gain_df.head(5)[["feature", "mean_gain"]].to_numpy()))
    total = time.perf_counter() - t_start
    print(f"  {total:.0f}s total, folds where xgb>hgb/hgb>xgb/tie: "
          f"{int((folds.xgb_acc > folds.hgb_acc).sum())}/{int((folds.hgb_acc > folds.xgb_acc).sum())}/{int((folds.xgb_acc == folds.hgb_acc).sum())}")

    preds.to_parquet(MODELS_DIR / f"loso_predictions_{name}.parquet", index=False)
    folds.to_parquet(MODELS_DIR / f"loso_folds_{name}.parquet", index=False)
    gain_df.to_parquet(MODELS_DIR / f"xgb_gain_{name}.parquet", index=False)
    write_json(MODELS_DIR / f"loso_{name}_manifest.json", {
        "run": name, "note": note,
        "n_samples": int(len(df)), "n_subjects": int(len(subjects)), "n_features": len(feature_cols),
        "labels": labels, "label_map": label_map, "feature_columns": feature_cols,
        "class_counts": {int(k): int(v) for k, v in pd.Series(y).value_counts().sort_index().items()},
        "normalization": "within-subject z-score, ddof=0, flat features set to 0, labels not used",
        "xgb_params": XGB_PARAMS if len(labels) > 2 else XGB_BINARY_PARAMS, "hgb_params": HGB_PARAMS,
        "class_weighting": "none",
        "runtime_seconds": {"total": total, "mean_xgb_per_fold": float(folds["xgb_seconds"].mean()),
                            "mean_hgb_per_fold": float(folds["hgb_seconds"].mean())},
        "metrics": summary,
        "folds_xgb_better": int((folds.xgb_acc > folds.hgb_acc).sum()),
        "folds_hgb_better": int((folds.hgb_acc > folds.xgb_acc).sum()),
    })
    return summary

In [7]:
run_loso("full_template_5class", model_full, kept_cols,
         note="full BL1 template: every BL1 trial contributes to its own template (self-inclusion)")

full_template_5class: 8426 trials, 86 subjects, 175 features, 5 classes


  fold 10/86      27s  eta  3.5 min


  fold 20/86      55s  eta  3.0 min


  fold 30/86      83s  eta  2.6 min


  fold 40/86     110s  eta  2.1 min


  fold 50/86     138s  eta  1.7 min


  fold 60/86     169s  eta  1.2 min


  fold 70/86     196s  eta  0.7 min


  fold 80/86     223s  eta  0.3 min


  fold 86/86     240s  eta  0.0 min
  XGBoost  acc 0.6610  macro-F1 0.6599  kappa_q 0.8502  MAE 0.420  fold acc median 0.662
  HGB      acc 0.6625  macro-F1 0.6627  kappa_q 0.8495  MAE 0.419  fold acc median 0.670
  top-5 XGB gain: emg_trapezius_sim_corr 0.047, ecg_sim_corr 0.043, emg_corrugator_sim_corr 0.037, emg_trapezius_sim_mi 0.021, emg_corrugator_ptp 0.020
  240s total, folds where xgb>hgb/hgb>xgb/tie: 38/40/8


{'xgb': {'accuracy': 0.66104913363399,
  'macro_f1': 0.659884583142452,
  'kappa_quadratic': 0.8501870127177711,
  'kappa_unweighted': 0.5763282988713923,
  'mae': 0.42036553524804177,
  'spearman': 0.8525475055765775,
  'within_one_level': 0.9255874673629243},
 'hgb': {'accuracy': 0.6624732969380489,
  'macro_f1': 0.6627273253114675,
  'kappa_quadratic': 0.8494585749317458,
  'kappa_unweighted': 0.5781024527372532,
  'mae': 0.4194160930453359,
  'spearman': 0.8508547097410029,
  'within_one_level': 0.9252314265369096}}

In [8]:
run_loso("loo_5class", model_loo, kept_cols,
         note="leave-one-out template, all 175 features; the PA1 facial-EMG copy of BL1 still leaks through 6 similarity columns")

loo_5class: 8426 trials, 86 subjects, 175 features, 5 classes


  fold 10/86      28s  eta  3.5 min


  fold 20/86      55s  eta  3.0 min


  fold 30/86      82s  eta  2.5 min


  fold 40/86     109s  eta  2.1 min


  fold 50/86     136s  eta  1.6 min


  fold 60/86     163s  eta  1.2 min


  fold 70/86     190s  eta  0.7 min


  fold 80/86     216s  eta  0.3 min


  fold 86/86     232s  eta  0.0 min
  XGBoost  acc 0.5221  macro-F1 0.5192  kappa_q 0.5152  MAE 0.875  fold acc median 0.500
  HGB      acc 0.5176  macro-F1 0.5171  kappa_q 0.5086  MAE 0.878  fold acc median 0.510
  top-5 XGB gain: emg_corrugator_sim_corr 0.055, emg_zygomaticus_sim_corr 0.024, gsr_phasic_max 0.023, emg_corrugator_ptp 0.021, emg_corrugator_log_rms 0.019
  232s total, folds where xgb>hgb/hgb>xgb/tie: 44/37/5


{'xgb': {'accuracy': 0.5220745312129124,
  'macro_f1': 0.5191619025410218,
  'kappa_quadratic': 0.5151675559552651,
  'kappa_unweighted': 0.40258843293734015,
  'mae': 0.8753857108948493,
  'spearman': 0.5149370604774552,
  'within_one_level': 0.7362924281984334},
 'hgb': {'accuracy': 0.5175646807500593,
  'macro_f1': 0.5171077342766012,
  'kappa_quadratic': 0.5085716949759213,
  'kappa_unweighted': 0.3969248381813488,
  'mae': 0.8781153572276288,
  'spearman': 0.5102945326148041,
  'within_one_level': 0.7386660337051982}}

In [9]:
run_loso("loo_5class_169", model_loo, [c for c in kept_cols if c not in FACIAL_SIM_COLS],
         note="leave-one-out template without the six facial similarity columns: leakage-controlled 5-class estimate")

loo_5class_169: 8426 trials, 86 subjects, 169 features, 5 classes


  fold 10/86      26s  eta  3.3 min


  fold 20/86      53s  eta  2.9 min


  fold 30/86      79s  eta  2.5 min


  fold 40/86     105s  eta  2.0 min


  fold 50/86     132s  eta  1.6 min


  fold 60/86     158s  eta  1.1 min


  fold 70/86     185s  eta  0.7 min


  fold 80/86     211s  eta  0.3 min


  fold 86/86     227s  eta  0.0 min
  XGBoost  acc 0.3604  macro-F1 0.3541  kappa_q 0.4513  MAE 1.070  fold acc median 0.354
  HGB      acc 0.3509  macro-F1 0.3465  kappa_q 0.4307  MAE 1.091  fold acc median 0.333
  top-5 XGB gain: emg_corrugator_log_rms 0.026, emg_corrugator_ptp 0.022, gsr_phasic_max 0.020, gsr_raw_slope 0.015, emg_zygomaticus_ptp 0.015
  227s total, folds where xgb>hgb/hgb>xgb/tie: 43/30/13


{'xgb': {'accuracy': 0.3604319962022312,
  'macro_f1': 0.35406369631727935,
  'kappa_quadratic': 0.45133804201367134,
  'kappa_unweighted': 0.20041441080377098,
  'mae': 1.070140042724899,
  'spearman': 0.4583788179692217,
  'within_one_level': 0.7091146451459768},
 'hgb': {'accuracy': 0.3509375741751721,
  'macro_f1': 0.34647740013330663,
  'kappa_quadratic': 0.43067021719906107,
  'kappa_unweighted': 0.18852371373160948,
  'mae': 1.0910277711844292,
  'spearman': 0.4377495524794059,
  'within_one_level': 0.7010443864229765}}

In [10]:
run_loso("loo_4class_pa1ex", model_loo, kept_cols, label_map=FOUR_CLASS["label_map"],
         note="PA1 excluded (its facial EMG duplicates BL1); classes relabelled 0..3 = BL1, PA2, PA3, PA4")

loo_4class_pa1ex: 6755 trials, 86 subjects, 175 features, 4 classes


  fold 10/86      20s  eta  2.5 min


  fold 20/86      40s  eta  2.2 min


  fold 30/86      60s  eta  1.9 min


  fold 40/86      79s  eta  1.5 min


  fold 50/86      99s  eta  1.2 min


  fold 60/86     119s  eta  0.9 min


  fold 70/86     139s  eta  0.5 min


  fold 80/86     158s  eta  0.2 min


  fold 86/86     170s  eta  0.0 min
  XGBoost  acc 0.4401  macro-F1 0.4368  kappa_q 0.4867  MAE 0.783  fold acc median 0.438
  HGB      acc 0.4346  macro-F1 0.4320  kappa_q 0.4848  MAE 0.787  fold acc median 0.413
  top-5 XGB gain: gsr_phasic_max 0.023, emg_corrugator_ptp 0.023, emg_corrugator_log_rms 0.022, emg_zygomaticus_ptp 0.017, gsr_raw_slope 0.015
  170s total, folds where xgb>hgb/hgb>xgb/tie: 41/36/9


{'xgb': {'accuracy': 0.4401184307920059,
  'macro_f1': 0.43677822637617736,
  'kappa_quadratic': 0.4867333683071564,
  'kappa_unweighted': 0.25339106603389505,
  'mae': 0.7825314581791266,
  'spearman': 0.49069045166070496,
  'within_one_level': 0.814359733530718},
 'hgb': {'accuracy': 0.43464100666173205,
  'macro_f1': 0.432040241498137,
  'kappa_quadratic': 0.4847543712601603,
  'kappa_unweighted': 0.2461141513501528,
  'mae': 0.7872686898593634,
  'spearman': 0.4874348301623355,
  'within_one_level': 0.813619541080681}}

In [11]:
run_loso("loo_5class_nofacial", model_loo, NOFACIAL_COLS,
         note=f"leave-one-out template, no facial EMG ({len(NOFACIAL_COLS)} ECG, EDA and trapezius features)")

loo_5class_nofacial: 8426 trials, 86 subjects, 113 features, 5 classes


  fold 10/86      20s  eta  2.6 min


  fold 20/86      41s  eta  2.2 min


  fold 30/86      61s  eta  1.9 min


  fold 40/86      81s  eta  1.5 min


  fold 50/86     101s  eta  1.2 min


  fold 60/86     121s  eta  0.9 min


  fold 70/86     141s  eta  0.5 min


  fold 80/86     161s  eta  0.2 min


  fold 86/86     174s  eta  0.0 min
  XGBoost  acc 0.3271  macro-F1 0.3200  kappa_q 0.3918  MAE 1.158  fold acc median 0.313
  HGB      acc 0.3202  macro-F1 0.3154  kappa_q 0.3778  MAE 1.170  fold acc median 0.312
  top-5 XGB gain: gsr_raw_slope 0.032, gsr_phasic_max 0.029, gsr_raw_skew 0.025, gsr_raw_std 0.017, gsr_phasic_p75 0.016
  174s total, folds where xgb>hgb/hgb>xgb/tie: 43/34/9


{'xgb': {'accuracy': 0.3270828388321861,
  'macro_f1': 0.3200165311280556,
  'kappa_quadratic': 0.3917880325623747,
  'kappa_unweighted': 0.1587648780213633,
  'mae': 1.1580821267505341,
  'spearman': 0.395395249671984,
  'within_one_level': 0.684429147875623},
 'hgb': {'accuracy': 0.3201993828625682,
  'macro_f1': 0.3153818550705427,
  'kappa_quadratic': 0.37775391430891625,
  'kappa_unweighted': 0.15014723065017643,
  'mae': 1.1703061951103726,
  'spearman': 0.3817665687477047,
  'within_one_level': 0.6809874198908141}}

In [12]:
run_loso("loo_4class_nofacial", model_loo, NOFACIAL_COLS, label_map=FOUR_CLASS["label_map"],
         note=f"PA1 excluded, no facial EMG ({len(NOFACIAL_COLS)} ECG, EDA and trapezius features)")

loo_4class_nofacial: 6755 trials, 86 subjects, 113 features, 4 classes


  fold 10/86      15s  eta  1.9 min


  fold 20/86      30s  eta  1.7 min


  fold 30/86      45s  eta  1.4 min


  fold 40/86      61s  eta  1.2 min


  fold 50/86      76s  eta  0.9 min


  fold 60/86      91s  eta  0.7 min


  fold 70/86     107s  eta  0.4 min


  fold 80/86     122s  eta  0.2 min


  fold 86/86     131s  eta  0.0 min
  XGBoost  acc 0.4084  macro-F1 0.4044  kappa_q 0.4271  MAE 0.852  fold acc median 0.400
  HGB      acc 0.4040  macro-F1 0.4016  kappa_q 0.4098  MAE 0.862  fold acc median 0.392
  top-5 XGB gain: gsr_phasic_max 0.035, gsr_raw_slope 0.028, gsr_raw_skew 0.025, gsr_hjorth_activity 0.017, gsr_phasic_p75 0.016
  131s total, folds where xgb>hgb/hgb>xgb/tie: 46/29/11


{'xgb': {'accuracy': 0.4084381939304219,
  'macro_f1': 0.40444889341169676,
  'kappa_quadratic': 0.4271065758792676,
  'kappa_unweighted': 0.21119044929857322,
  'mae': 0.8519615099925981,
  'spearman': 0.4294367732735515,
  'within_one_level': 0.787120651369356},
 'hgb': {'accuracy': 0.40399703923019986,
  'macro_f1': 0.4015838181784138,
  'kappa_quadratic': 0.4098426558724655,
  'kappa_unweighted': 0.20525412605140292,
  'mae': 0.8615840118430792,
  'spearman': 0.4119149281689138,
  'within_one_level': 0.7853441894892672}}

## Step 5 — evaluation

Accuracy, macro-F1, quadratic and unweighted kappa, MAE, Spearman and within-one-level rate, with
subject-level bootstrap CIs (2000 resamples) and a paired bootstrap for XGBoost minus HGB. Binary tasks are
derived from the multiclass probabilities. The last block quantifies the self-inclusion effect by comparing
the full-template run with the leave-one-out run.

In [13]:
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, log_loss,
                             precision_recall_fscore_support, roc_auc_score)

N_BOOT = 2000
CORE_KEYS = ["accuracy", "macro_f1", "kappa_quadratic", "kappa_unweighted", "mae", "spearman", "within_one_level"]
TASKS = {
    "loo_5class": dict(label="5-class, leave-one-out template, 175 features", names=CLASS_NAMES),
    "loo_5class_169": dict(label="5-class, leave-one-out template, 169 features (no facial similarity)", names=CLASS_NAMES),
    "loo_4class_pa1ex": dict(label="4-class, PA1 excluded, 175 features", names=FOUR_CLASS["names"]),
    "loo_5class_nofacial": dict(label="5-class, leave-one-out template, no facial EMG", names=CLASS_NAMES),
    "loo_4class_nofacial": dict(label="4-class, PA1 excluded, no facial EMG", names=FOUR_CLASS["names"]),
}


def load_predictions(name):
    preds = pd.read_parquet(MODELS_DIR / f"loso_predictions_{name}.parquet")
    labels = sorted(int(c[len("prob_"):]) for c in preds.columns if c.startswith("prob_"))
    blocks = {m: preds[preds["model"] == m].sort_values("sample_id").reset_index(drop=True) for m in MODELS}
    ref = blocks["xgb"]["sample_id"].to_numpy()
    assert all(np.array_equal(blocks[m]["sample_id"].to_numpy(), ref) for m in MODELS)
    return blocks, labels


def prob_metrics(y_true, prob, labels):
    onehot = np.zeros_like(prob)
    onehot[np.arange(len(y_true)), y_true] = 1.0
    return {"log_loss": float(log_loss(y_true, prob, labels=labels)),
            "brier": float(np.mean(np.sum((prob - onehot) ** 2, axis=1))),
            "mean_max_prob": float(prob.max(axis=1).mean())}


def binary_metrics(y_bin, score, threshold=0.5):
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_bin, pred, labels=[0, 1]).ravel()
    return {"n_samples": int(len(y_bin)), "accuracy": float(accuracy_score(y_bin, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_bin, pred)), "auc": float(roc_auc_score(y_bin, score)),
            "sensitivity": float(tp / (tp + fn)) if tp + fn else np.nan,
            "specificity": float(tn / (tn + fp)) if tn + fp else np.nan}


def subject_bootstrap(subject, y_true, preds_by_model, labels, n_boot=N_BOOT, seed=SEED):
    """Resample subjects; returns per-model CIs and the paired XGB-minus-HGB distribution."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(subject)
    idx_map = {s: np.flatnonzero(subject == s) for s in uniq}
    draws = {m: {k: [] for k in CORE_KEYS} for m in MODELS}
    for _ in range(n_boot):
        idx = np.concatenate([idx_map[s] for s in rng.choice(uniq, size=len(uniq), replace=True)])
        for m in MODELS:
            mm = core_metrics(y_true[idx], preds_by_model[m][idx], labels)
            for k in CORE_KEYS:
                draws[m][k].append(mm[k])
    ci = {m: {k: {"mean": float(np.mean(v)), "ci95": [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]}
              for k, v in draws[m].items()} for m in MODELS}
    paired = {}
    for k in CORE_KEYS:
        d = np.asarray(draws["xgb"][k]) - np.asarray(draws["hgb"][k])
        paired[k] = {"mean_delta": float(d.mean()), "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
                     "p_two_sided": float(min(1.0, 2 * min(np.mean(d > 0), np.mean(d < 0))))}
    return ci, paired


def evaluate_task(key, spec):
    blocks, labels = load_predictions(key)
    names = spec["names"]
    y_true = blocks["xgb"]["true_class"].to_numpy(dtype=int)
    subject = blocks["xgb"]["subject_id"].to_numpy(dtype=int)
    y_pred = {m: blocks[m]["pred_class"].to_numpy(dtype=int) for m in MODELS}
    prob_cols = [f"prob_{j}" for j in labels]
    prob = {m: blocks[m][prob_cols].to_numpy(dtype=np.float64) for m in MODELS}
    prob = {m: p / p.sum(axis=1, keepdims=True) for m, p in prob.items()}       # float32 rounding

    print("\n" + "#" * 78)
    print(f"# {key}: {spec['label']}   ({len(y_true)} trials, {len(np.unique(subject))} subjects)")
    print("#" * 78)
    report = {"task": key, "label": spec["label"], "n_samples": int(len(y_true)), "n_subjects": int(len(np.unique(subject))),
              "class_counts": {names[k]: int(v) for k, v in pd.Series(y_true).value_counts().sort_index().items()},
              "models": {}}
    for m in MODELS:
        cm = confusion_matrix(y_true, y_pred[m], labels=labels)
        precision, recall, f1c, support = precision_recall_fscore_support(y_true, y_pred[m], labels=labels, zero_division=0)
        per_class = pd.DataFrame({"class": [names[k] for k in labels], "support": support.astype(int), "precision": precision,
                                  "recall": recall, "f1": f1c,
                                  "auc_ovr": [roc_auc_score((y_true == k).astype(int), prob[m][:, k]) for k in labels]})
        pain = y_true != 0
        binary = {"BL1_vs_any_pain": binary_metrics(pain.astype(int), 1.0 - prob[m][:, 0])}
        for k in labels[1:]:
            mask = np.isin(y_true, [0, k])
            pair = prob[m][mask, k] / (prob[m][mask, 0] + prob[m][mask, k] + 1e-12)
            binary[f"BL1_vs_{names[k]}"] = binary_metrics((y_true[mask] == k).astype(int), pair)
        report["models"][m] = {
            "name": MODEL_NAMES[m],
            "core": core_metrics(y_true, y_pred[m], labels),
            "probability": prob_metrics(y_true, prob[m], labels),
            "confusion_counts": cm.tolist(),
            "confusion_row_normalised": (cm / cm.sum(axis=1, keepdims=True)).tolist(),
            "per_class": per_class.to_dict("records"),
            "binary": binary,
            "pain_only": core_metrics(y_true[pain], y_pred[m][pain], labels[1:]),
        }
        print(f"\n{MODEL_NAMES[m]} confusion (rows = true)")
        print(pd.DataFrame(cm, index=[names[k] for k in labels], columns=[names[k] for k in labels]).to_string())
        print(per_class.round(3).to_string(index=False))
        print("binary tasks:")
        print(pd.DataFrame(binary).T.round(3).to_string())

    ci, paired = subject_bootstrap(subject, y_true, y_pred, labels)
    for m in MODELS:
        report["models"][m]["bootstrap"] = ci[m]
    report["paired_xgb_minus_hgb"] = paired

    head = pd.DataFrame({MODEL_NAMES[m]: report["models"][m]["core"] for m in MODELS}).T
    print("\ncore metrics")
    print(head.round(4).to_string())
    print("\naccuracy 95% CI:", {MODEL_NAMES[m]: [round(v, 4) for v in ci[m]["accuracy"]["ci95"]] for m in MODELS},
          "  XGB-HGB: %+.4f [%+.4f, %+.4f] p=%.3f" % (paired["accuracy"]["mean_delta"], *paired["accuracy"]["ci95"],
                                                     paired["accuracy"]["p_two_sided"]))

    # ---- per subject ----
    rows = []
    for sid in np.unique(subject):
        mask = subject == sid
        row = {"subject_id": int(sid), "n_samples": int(mask.sum())}
        for m in MODELS:
            yt, yp = y_true[mask], y_pred[m][mask]
            row[f"acc_{m}"] = float(accuracy_score(yt, yp))
            row[f"mae_{m}"] = float(np.mean(np.abs(yt - yp)))
            row[f"bl1_recall_{m}"] = float(np.mean(yp[yt == 0] == 0)) if (yt == 0).any() else np.nan
            row[f"pain_acc_{m}"] = float(accuracy_score(yt[yt != 0], yp[yt != 0])) if (yt != 0).any() else np.nan
        rows.append(row)
    per_subject = pd.DataFrame(rows)
    per_subject.to_parquet(REPORTS_DIR / f"per_subject_{key}.parquet", index=False)

    # ---- markdown ----
    md = [f"# BioVid Part B evaluation: {spec['label']}\n",
          f"LOSO over **{report['n_subjects']} subjects / {report['n_samples']} trials**, unweighted classes, identical folds.\n",
          "## Core results\n",
          "| model | accuracy | macro-F1 | kappa (quadratic) | kappa (unweighted) | MAE | Spearman | within 1 level |",
          "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for m in MODELS:
        c = report["models"][m]["core"]
        md.append(f"| {MODEL_NAMES[m]} | " + " | ".join(f"{c[k]:.4f}" for k in CORE_KEYS) + " |")
    md += ["\n## Subject-level bootstrap 95% CIs (2000 resamples)\n", "| model | metric | point | 95% CI |", "|---|---|---:|---|"]
    for m in MODELS:
        for k in CORE_KEYS:
            md.append(f"| {MODEL_NAMES[m]} | {k} | {report['models'][m]['core'][k]:.4f} | [{ci[m][k]['ci95'][0]:.4f}, {ci[m][k]['ci95'][1]:.4f}] |")
    md += ["\n## Paired bootstrap: XGBoost - HGB\n", "| metric | delta | 95% CI | p (two-sided) |", "|---|---:|---|---:|"]
    for k in CORE_KEYS:
        d = paired[k]
        md.append(f"| {k} | {d['mean_delta']:+.4f} | [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] | {d['p_two_sided']:.3f} |")
    md += ["\n## Binary tasks derived from the multiclass probabilities\n",
           "| task | model | n | accuracy | balanced acc | AUC | sensitivity | specificity |", "|---|---|---:|---:|---:|---:|---:|---:|"]
    for m in MODELS:
        for task, v in report["models"][m]["binary"].items():
            md.append(f"| {task} | {MODEL_NAMES[m]} | {v['n_samples']} | {v['accuracy']:.4f} | {v['balanced_accuracy']:.4f} | "
                      f"{v['auc']:.4f} | {v['sensitivity']:.4f} | {v['specificity']:.4f} |")
    md += ["\n## Per-class metrics\n", "| model | class | support | precision | recall | F1 | AUC (one-vs-rest) |", "|---|---|---:|---:|---:|---:|---:|"]
    for m in MODELS:
        for r in report["models"][m]["per_class"]:
            md.append(f"| {MODEL_NAMES[m]} | {r['class']} | {r['support']} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1']:.4f} | {r['auc_ovr']:.4f} |")
    md.append("\n## Confusion matrices (rows = true)\n")
    for m in MODELS:
        md += [f"\n**{MODEL_NAMES[m]}**\n", "| true \\ pred | " + " | ".join(names[k] for k in labels) + " |",
               "|---|" + "|".join("---:" for _ in labels) + "|"]
        for i, k in enumerate(labels):
            md.append(f"| {names[k]} | " + " | ".join(str(v) for v in report["models"][m]["confusion_counts"][i]) + " |")
    md.append("\nAll CIs resample subjects, not trials. The BL1-vs-any-pain accuracy uses a 0.5 threshold on 1 - P(BL1); "
              "pain trials are 80% of the sample, so AUC and balanced accuracy are the informative numbers.\n")

    write_json(REPORTS_DIR / f"eval_{key}.json", report)
    (REPORTS_DIR / f"eval_{key}.md").write_text("\n".join(md), encoding="utf-8")
    return report


reports = {key: evaluate_task(key, spec) for key, spec in TASKS.items()}

# ---------------- self-inclusion effect: full template vs leave-one-out ----------------
print("\n" + "#" * 78)
print("# SELF-INCLUSION EFFECT: full BL1 template vs leave-one-out template (5-class, 175 features)")
print("#" * 78)
blocks_full, _ = load_predictions("full_template_5class")
y_full = blocks_full["xgb"]["true_class"].to_numpy(dtype=int)
comparison = []
for m in MODELS:
    a = core_metrics(y_full, blocks_full[m]["pred_class"].to_numpy(dtype=int), LABELS)
    b = reports["loo_5class"]["models"][m]["core"]
    c = reports["loo_5class_169"]["models"][m]["core"]
    comparison.append({"model": MODEL_NAMES[m], "acc_full_template": a["accuracy"], "acc_loo": b["accuracy"],
                       "acc_loo_169": c["accuracy"], "kappa_q_full_template": a["kappa_quadratic"],
                       "kappa_q_loo": b["kappa_quadratic"], "kappa_q_loo_169": c["kappa_quadratic"]})
comparison = pd.DataFrame(comparison)
print(comparison.round(4).to_string(index=False))

# How separable is BL1 from a single similarity feature before and after the correction?
model_full = pd.read_parquet(FEATURES_DIR / "features_model_partB.parquet").sort_values("sample_id").reset_index(drop=True)
model_loo = pd.read_parquet(FEATURES_DIR / "features_model_partB_loo.parquet").sort_values("sample_id").reset_index(drop=True)
is_bl1 = (model_loo["class_id"].to_numpy() == 0).astype(int)
subj = model_loo["subject_id"].to_numpy()
diag = []
for c in ["gsr_sim_corr", "ecg_sim_corr", "emg_trapezius_sim_corr", "emg_corrugator_sim_corr", "emg_zygomaticus_sim_corr"]:
    z_full, _ = zscore_within_subject(model_full[[c]].to_numpy(), subj)
    z_loo, _ = zscore_within_subject(model_loo[[c]].to_numpy(), subj)
    diag.append({"feature": c, "auc_bl1_full_template": float(roc_auc_score(is_bl1, z_full[:, 0])),
                 "auc_bl1_loo_template": float(roc_auc_score(is_bl1, z_loo[:, 0]))})
diag = pd.DataFrame(diag)
print("\nBL1-vs-rest AUC of single similarity features (within-subject z-scored):")
print(diag.round(4).to_string(index=False))

write_json(REPORTS_DIR / "template_correction.json", {
    "full_template_vs_loo": comparison.to_dict("records"),
    "similarity_feature_auc": diag.to_dict("records"),
    "note": "the full template lets each BL1 trial match itself; leave-one-out removes that, the 169-feature run also removes the PA1 facial copy",
})


##############################################################################
# loo_5class: 5-class, leave-one-out template, 175 features   (8426 trials, 86 subjects)
##############################################################################

XGBoost confusion (rows = true)
     BL1   PA1  PA2  PA3   PA4
BL1  791    63  372  312   130
PA1   73  1501   56   32     9
PA2  491    55  500  413   227
PA3  310    40  377  549   424
PA4  107    22  170  344  1058
class  support  precision  recall    f1  auc_ovr
  BL1     1668      0.446   0.474 0.460    0.800
  PA1     1671      0.893   0.898 0.896    0.985
  PA2     1686      0.339   0.297 0.316    0.709
  PA3     1700      0.333   0.323 0.328    0.709
  PA4     1701      0.573   0.622 0.596    0.865
binary tasks:
                 n_samples  accuracy  balanced_accuracy    auc  sensitivity  specificity
BL1_vs_any_pain     8426.0     0.817              0.577  0.800        0.974        0.181
BL1_vs_PA1          3339.0     0.932           


core metrics
         accuracy  macro_f1  kappa_quadratic  kappa_unweighted     mae  spearman  within_one_level
XGBoost    0.5221    0.5192           0.5152            0.4026  0.8754    0.5149            0.7363
HGB        0.5176    0.5171           0.5086            0.3969  0.8781    0.5103            0.7387

accuracy 95% CI: {'XGBoost': [0.4979, 0.5469], 'HGB': [0.4936, 0.5418]}   XGB-HGB: +0.0045 [-0.0019, +0.0107] p=0.169

##############################################################################
# loo_5class_169: 5-class, leave-one-out template, 169 features (no facial similarity)   (8426 trials, 86 subjects)
##############################################################################

XGBoost confusion (rows = true)
     BL1  PA1  PA2  PA3   PA4
BL1  631  359  278  273   127
PA1  383  479  343  308   158
PA2  323  424  323  391   225
PA3  219  225  292  555   409
PA4   81  111  123  337  1049
class  support  precision  recall    f1  auc_ovr
  BL1     1668      0.385   0.378


core metrics
         accuracy  macro_f1  kappa_quadratic  kappa_unweighted     mae  spearman  within_one_level
XGBoost    0.3604    0.3541           0.4513            0.2004  1.0701    0.4584            0.7091
HGB        0.3509    0.3465           0.4307            0.1885  1.0910    0.4377            0.7010

accuracy 95% CI: {'XGBoost': [0.3357, 0.3849], 'HGB': [0.3269, 0.3759]}   XGB-HGB: +0.0094 [+0.0014, +0.0173] p=0.020

##############################################################################
# loo_4class_pa1ex: 4-class, PA1 excluded, 175 features   (6755 trials, 86 subjects)
##############################################################################

XGBoost confusion (rows = true)
     BL1  PA2  PA3   PA4
BL1  779  409  338   142
PA2  476  520  460   230
PA3  278  396  600   426
PA4  108  158  361  1074
class  support  precision  recall    f1  auc_ovr
  BL1     1668      0.475   0.467 0.471    0.757
  PA2     1686      0.351   0.308 0.328    0.643
  PA3     1700      0


core metrics
         accuracy  macro_f1  kappa_quadratic  kappa_unweighted     mae  spearman  within_one_level
XGBoost    0.4401    0.4368           0.4867            0.2534  0.7825    0.4907            0.8144
HGB        0.4346    0.4320           0.4848            0.2461  0.7873    0.4874            0.8136

accuracy 95% CI: {'XGBoost': [0.4133, 0.4677], 'HGB': [0.4058, 0.4638]}   XGB-HGB: +0.0055 [-0.0043, +0.0154] p=0.270

##############################################################################
# loo_5class_nofacial: 5-class, leave-one-out template, no facial EMG   (8426 trials, 86 subjects)
##############################################################################

XGBoost confusion (rows = true)
     BL1  PA1  PA2  PA3  PA4
BL1  610  385  241  251  181
PA1  409  452  326  267  217
PA2  328  411  283  395  269
PA3  230  277  315  456  422
PA4  112  145  141  348  955
class  support  precision  recall    f1  auc_ovr
  BL1     1668      0.361   0.366 0.363    0.697
  PA1  


core metrics
         accuracy  macro_f1  kappa_quadratic  kappa_unweighted     mae  spearman  within_one_level
XGBoost    0.3271    0.3200           0.3918            0.1588  1.1581    0.3954            0.6844
HGB        0.3202    0.3154           0.3778            0.1501  1.1703    0.3818            0.6810

accuracy 95% CI: {'XGBoost': [0.305, 0.3504], 'HGB': [0.2981, 0.3429]}   XGB-HGB: +0.0067 [-0.0005, +0.0148] p=0.062

##############################################################################
# loo_4class_nofacial: 4-class, PA1 excluded, no facial EMG   (6755 trials, 86 subjects)
##############################################################################

XGBoost confusion (rows = true)
     BL1  PA2  PA3  PA4
BL1  765  403  311  189
PA2  456  503  453  274
PA3  332  420  494  454
PA4  132  200  372  997
class  support  precision  recall    f1  auc_ovr
  BL1     1668      0.454   0.459 0.456    0.739
  PA2     1686      0.330   0.298 0.313    0.616
  PA3     1700      0.3


core metrics
         accuracy  macro_f1  kappa_quadratic  kappa_unweighted     mae  spearman  within_one_level
XGBoost    0.4084    0.4044           0.4271            0.2112  0.8520    0.4294            0.7871
HGB        0.4040    0.4016           0.4098            0.2053  0.8616    0.4119            0.7853

accuracy 95% CI: {'XGBoost': [0.379, 0.439], 'HGB': [0.3762, 0.4323]}   XGB-HGB: +0.0046 [-0.0064, +0.0150] p=0.374

##############################################################################
# SELF-INCLUSION EFFECT: full BL1 template vs leave-one-out template (5-class, 175 features)
##############################################################################
  model  acc_full_template  acc_loo  acc_loo_169  kappa_q_full_template  kappa_q_loo  kappa_q_loo_169
XGBoost             0.6610   0.5221       0.3604                 0.8502       0.5152           0.4513
    HGB             0.6625   0.5176       0.3509                 0.8495       0.5086           0.4307

BL1-vs-rest A

## Step 6 — PA1 facial-EMG duplication audit

Hashes of the cached corrugator and zygomaticus arrays show how many PA1 trials are byte-identical to a BL1 trial
of the same subject. The twin-pair block checks how the leave-one-out 5-class model treats such pairs, and the
source-file check confirms the duplication exists in the released CSVs, not in the cache. Hashes cannot tell which
class holds the original; Step 9 settles that with the features Gruss et al. computed from the original recordings.

In [14]:
import hashlib

FACIAL = {"emg_corrugator": 3, "emg_zygomaticus": 4}
digest = lambda a: hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()

meta = pd.read_parquet(CACHE_DIR / "meta.parquet").sort_values("sample_id").reset_index(drop=True)
preds = pd.read_parquet(MODELS_DIR / "loso_predictions_loo_5class.parquet")

audit_rows, pairs = [], []
for sid, g in meta.groupby("subject_id", sort=True):
    g = g.sort_values("cache_row").reset_index(drop=True)
    arr = np.load(CACHE_DIR / g["cache_file"].iloc[0])
    y = g["class_id"].to_numpy()
    row = {"subject_id": int(sid), "subject_name": g["subject_name"].iloc[0], "n_trials": int(len(g)),
           "n_bl1": int((y == 0).sum()), "n_pa1": int((y == 1).sum())}
    for name, ci in FACIAL.items():
        h = np.array([digest(a) for a in arr[:, ci, :]])
        h0, h1 = set(h[y == 0]), set(h[y == 1])
        row[f"{name}_n_unique"] = len(set(h))
        row[f"{name}_set_equal"] = h0 == h1
        row[f"{name}_pa1_matched"] = len(h0 & h1)
        row[f"{name}_pa1_frac"] = len(h0 & h1) / len(h1) if h1 else np.nan
        a0, a1 = arr[y == 0, ci, :], arr[y == 1, ci, :]
        row[f"{name}_order_exact"] = bool(np.array_equal(a0, a1)) if a0.shape == a1.shape else None
        if name == "emg_corrugator":                           # twin pairs on the corrugator hash
            bl1 = g[y == 0].assign(h=h[y == 0])
            pa1 = g[y == 1].assign(h=h[y == 1])
            for _, r in bl1.merge(pa1, on="h", suffixes=("_bl1", "_pa1")).iterrows():
                pairs.append({"subject_id": int(sid), "sample_id_bl1": int(r["sample_id_bl1"]), "sample_id_pa1": int(r["sample_id_pa1"])})
    audit_rows.append(row)
audit = pd.DataFrame(audit_rows)
pairs = pd.DataFrame(pairs)

print("PART 1 - duplication in the released filtered signals")
per_channel = {}
for name in FACIAL:
    per_channel[name] = {
        "subjects_set_equal": int(audit[f"{name}_set_equal"].sum()),
        "subjects_pa1_fully_matched": int((audit[f"{name}_pa1_frac"] >= 0.999).sum()),
        "subjects_pa1_zero_matched": int((audit[f"{name}_pa1_frac"] <= 0.001).sum()),
        "min_pa1_match_fraction": float(audit[f"{name}_pa1_frac"].min()),
        "pooled_pa1_matched": int(audit[f"{name}_pa1_matched"].sum()),
        "pooled_pa1_total": int(audit["n_pa1"].sum()),
        "order_aligned_exact_subjects": int(audit[f"{name}_order_exact"].astype(bool).sum()),
        "comparable_subjects": int(audit[f"{name}_order_exact"].notna().sum()),
    }
    v = per_channel[name]
    print(f"  {name:16s} BL1 set == PA1 set: {v['subjects_set_equal']}/{len(audit)}   PA1 fully matched: "
          f"{v['subjects_pa1_fully_matched']}/{len(audit)}   pooled PA1 matched: {v['pooled_pa1_matched']}/{v['pooled_pa1_total']} "
          f"({100 * v['pooled_pa1_matched'] / v['pooled_pa1_total']:.1f}%)   min match {v['min_pa1_match_fraction']:.3f}")
partial = audit[(audit["emg_corrugator_pa1_frac"] < 0.999) | (audit["emg_zygomaticus_pa1_frac"] < 0.999)]
print("\nsubjects with < 100% PA1 match on either facial channel:")
print(partial[["subject_name", "n_bl1", "n_pa1", "emg_corrugator_pa1_frac", "emg_zygomaticus_pa1_frac"]]
      .round(3).to_string(index=False))

print(f"\nPART 2 - {len(pairs)} matched BL1/PA1 twin pairs in {pairs['subject_id'].nunique()} subjects")
twin = {}
for m in MODELS:
    p = preds[preds["model"] == m].set_index("sample_id")
    a = p.loc[pairs["sample_id_bl1"]]
    b = p.loc[pairs["sample_id_pa1"]]
    pa, pb = a["pred_class"].to_numpy(), b["pred_class"].to_numpy()
    twin[m] = {"both_correct": float(np.mean((pa == 0) & (pb == 1))), "same_label": float(np.mean(pa == pb)),
               "pa1_prob_higher_for_pa1_twin": float(np.mean(b["prob_1"].to_numpy() > a["prob_1"].to_numpy()))}
    print(f"  {MODEL_NAMES[m]:<8s} both twins correct {twin[m]['both_correct']:.3f}   same label {twin[m]['same_label']:.3f}   "
          f"P(PA1) higher for the PA1 twin {twin[m]['pa1_prob_higher_for_pa1_twin']:.3f}")

print("\nPART 3 - source-file check on the first twin pair")
r0 = pairs.iloc[0]
names = meta.set_index("sample_id").loc[[r0["sample_id_bl1"], r0["sample_id_pa1"]]]
da, db = (pd.read_csv(p, sep="\t") for p in names["bio_path"])
source_check = {"files": names["sample_name"].tolist(),
                "byte_identical": {ch: bool(np.array_equal(da[ch].to_numpy(), db[ch].to_numpy())) for ch in CHANNELS}}
for ch, eq in source_check["byte_identical"].items():
    print(f"  {ch:16s} byte-identical: {eq}")

summary = {"n_trials": int(len(meta)), "n_subjects": int(meta["subject_id"].nunique()), "per_channel": per_channel,
           "n_twin_pairs": int(len(pairs)), "twin_pair_predictions_loo_5class": twin, "source_file_check": source_check}
write_json(REPORTS_DIR / "pa1_duplication.json", summary)
audit.to_parquet(REPORTS_DIR / "pa1_duplication_per_subject.parquet", index=False)
pairs.to_parquet(REPORTS_DIR / "pa1_twin_pairs.parquet", index=False)
md = ["# PA1 facial-EMG duplication audit\n",
      f"Retained trials: **{len(meta)}** across **{meta['subject_id'].nunique()}** subjects.\n",
      "| channel | subjects BL1 set == PA1 set | PA1 fully matched | pooled PA1 matched | min PA1 match | order-aligned exact |",
      "|---|---:|---:|---:|---:|---:|"]
for name, v in per_channel.items():
    md.append(f"| `{name}` | {v['subjects_set_equal']} / {len(audit)} | {v['subjects_pa1_fully_matched']} / {len(audit)} | "
              f"{v['pooled_pa1_matched']} / {v['pooled_pa1_total']} ({100 * v['pooled_pa1_matched'] / v['pooled_pa1_total']:.1f}%) | "
              f"{v['min_pa1_match_fraction']:.3f} | {v['order_aligned_exact_subjects']} / {v['comparable_subjects']} |")
md += [f"\n## Twin pairs ({len(pairs)}), leave-one-out 5-class model\n",
       "| model | both twins correct | same label | P(PA1) higher for the PA1 twin |", "|---|---:|---:|---:|"]
for m, v in twin.items():
    md.append(f"| {MODEL_NAMES[m]} | {v['both_correct']:.4f} | {v['same_label']:.4f} | {v['pa1_prob_higher_for_pa1_twin']:.4f} |")
md += ["\n## Source-file check\n", f"`{source_check['files'][0]}` vs `{source_check['files'][1]}`:\n", "| channel | byte-identical |", "|---|---|"]
md += [f"| `{ch}` | {'yes' if eq else 'no'} |" for ch, eq in source_check["byte_identical"].items()]
md.append("\nByte-identical facial signals across two experimental conditions cannot be physiological. "
          "With a leave-one-out template a PA1 trial still finds its BL1 twin inside the template while the BL1 trial "
          "cannot find itself, so the facial similarity features become a PA1 detector. They carry no BL1-vs-PA1 information. "
          "Which class holds the original recording is established in Step 9 (`published_features_audit.json`).\n")
(REPORTS_DIR / "pa1_duplication.md").write_text("\n".join(md), encoding="utf-8")
print("\nwrote:", REPORTS_DIR / "pa1_duplication.md")

PART 1 - duplication in the released filtered signals
  emg_corrugator   BL1 set == PA1 set: 72/86   PA1 fully matched: 75/86   pooled PA1 matched: 1656/1671 (99.1%)   min match 0.750
  emg_zygomaticus  BL1 set == PA1 set: 72/86   PA1 fully matched: 75/86   pooled PA1 matched: 1656/1671 (99.1%)   min match 0.750

subjects with < 100% PA1 match on either facial channel:
subject_name  n_bl1  n_pa1  emg_corrugator_pa1_frac  emg_zygomaticus_pa1_frac
 072609_w_23     19     20                    0.950                     0.950
 073109_w_28     19     20                    0.950                     0.950
 080314_w_25     19     20                    0.950                     0.950
 082909_m_47     19     20                    0.950                     0.950
 083009_w_42     19     19                    0.947                     0.947
 083114_w_55     15     12                    0.750                     0.750
 091914_m_46     18     18                    0.889                     0.889
 100

## Step 7 — figures

In [15]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COLORS = {"xgb": "#1f77b4", "hgb": "#ff7f0e"}
MOD_COLORS = {"gsr": "#2ca02c", "ecg": "#d62728", "emg": "#1f77b4"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "font.size": 9, "axes.titlesize": 10.5,
                     "axes.spines.top": False, "axes.spines.right": False})


def plot_confusion(ax, y_true, y_pred, names, title, cmap):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(names))))
    row = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(row, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{row[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7.5,
                    color="white" if row[i, j] > 0.55 else "black")


def confusion_figure(run, names, title, cmap, fname):
    blocks, _ = load_predictions(run)
    y = blocks["xgb"]["true_class"].to_numpy(dtype=int)
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0))
    for ax, m in zip(axes, MODELS):
        plot_confusion(ax, y, blocks[m]["pred_class"].to_numpy(dtype=int), names, f"{MODEL_NAMES[m]} - {title}", cmap)
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, bbox_inches="tight")
    plt.close(fig)


confusion_figure("loo_5class_169", list(CLASS_NAMES.values()), "5-class, 169 features", "Blues", "fig01_confusion_5class_169.png")
confusion_figure("loo_4class_pa1ex", list(FOUR_CLASS["names"].values()), "4-class, PA1 excluded", "Greens", "fig02_confusion_4class_pa1ex.png")

# ---- per-subject accuracy, leakage-controlled 5-class ----
per = pd.read_parquet(REPORTS_DIR / "per_subject_loo_5class_169.parquet").set_index("subject_id")
low_n = per.index[per["n_samples"] < 90]
meta = pd.read_parquet(CACHE_DIR / "meta.parquet")
names_by_id = meta.drop_duplicates("subject_id").set_index("subject_id")["subject_name"]
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
bp = ax.boxplot([per[f"acc_{m}"] for m in MODELS], tick_labels=[MODEL_NAMES[m] for m in MODELS], widths=0.5,
                patch_artist=True, showfliers=False)
for patch, m in zip(bp["boxes"], MODELS):
    patch.set_facecolor(COLORS[m])
    patch.set_alpha(0.35)
rng = np.random.default_rng(0)
for i, m in enumerate(MODELS, start=1):
    ax.scatter(i + rng.uniform(-0.09, 0.09, len(per)), per[f"acc_{m}"], s=9, color=COLORS[m], alpha=0.7, linewidths=0)
ax.set_ylabel("per-subject LOSO accuracy")
ax.set_title(f"5-class, 169 features: {len(per)} subjects")
ax.grid(axis="y", alpha=0.25)
ax = axes[1]
ax.scatter(per["acc_xgb"], per["acc_hgb"], s=18, alpha=0.75, color="#555555", linewidths=0)
lim = [0, float(per[["acc_xgb", "acc_hgb"]].max().max()) * 1.05]
ax.plot(lim, lim, "--", color="grey", lw=0.8)
for sid in low_n:                                                     # subjects that lost many trials
    ax.annotate(f"{names_by_id[sid]} (n={per.loc[sid, 'n_samples']})", (per.loc[sid, "acc_xgb"], per.loc[sid, "acc_hgb"]),
                fontsize=6.5, xytext=(5, -9), textcoords="offset points", color="#b22222")
ax.set_xlabel("XGBoost accuracy")
ax.set_ylabel("HGB accuracy")
ax.set_title("paired per-subject accuracy")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig03_per_subject_accuracy.png", bbox_inches="tight")
plt.close(fig)

# ---- XGB gain importance: 175 (with facial similarity) vs 169 features ----
fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.8))
for ax, run, title in zip(axes, ["loo_5class", "loo_5class_169"],
                          ["175 features (facial similarity = PA1-copy proxies)", "169 features (leakage-controlled)"]):
    gain = pd.read_parquet(MODELS_DIR / f"xgb_gain_{run}.parquet").head(20).iloc[::-1]
    ax.barh(gain["feature"], gain["mean_gain"], color=[MOD_COLORS[f.split("_")[0]] for f in gain["feature"]])
    ax.set_xlabel("mean XGB gain share across LOSO folds")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in MOD_COLORS.values()]
axes[1].legend(handles, ["EDA (GSR)", "ECG", "EMG"], frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig04_importance_top20.png", bbox_inches="tight")
plt.close(fig)

# ---- correction history: full template -> leave-one-out -> 169 features ----
stages = [("full_template_5class", "full template"), ("loo_5class", "leave-one-out"), ("loo_5class_169", "LOO, 169 features")]
metrics = [("accuracy", "accuracy"), ("macro_f1", "macro-F1"), ("kappa_quadratic", "quadratic kappa")]
scores = {}
for run, _ in stages:
    blocks, labels = load_predictions(run)
    y = blocks["xgb"]["true_class"].to_numpy(dtype=int)
    scores[run] = {m: core_metrics(y, blocks[m]["pred_class"].to_numpy(dtype=int), labels) for m in MODELS}
fig, ax = plt.subplots(figsize=(8.4, 3.9))
x = np.arange(len(metrics))
width = 0.13
for si, (run, stage) in enumerate(stages):
    for mi, m in enumerate(MODELS):
        offset = (si * 2 + mi - 2.5) * width
        vals = [scores[run][m][k] for k, _ in metrics]
        ax.bar(x + offset, vals, width * 0.92, color=COLORS[m], alpha=0.35 + 0.3 * si, label=f"{MODEL_NAMES[m]}, {stage}")
        for xi, v in zip(x + offset, vals):
            ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=6.3)
ax.set_xticks(x, [lbl for _, lbl in metrics])
ax.set_ylim(0, 1.0)
ax.set_title("5-class score at each correction stage")
ax.legend(frameon=False, fontsize=7, ncol=3)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig05_correction_history.png", bbox_inches="tight")
plt.close(fig)

for p in sorted(FIG_DIR.glob("fig*.png")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KiB)")

  fig01_confusion_5class_169.png  (104 KiB)
  fig02_confusion_4class_pa1ex.png  (81 KiB)
  fig03_per_subject_accuracy.png  (81 KiB)
  fig04_importance_top20.png  (142 KiB)
  fig05_correction_history.png  (37 KiB)


## Step 8 — linear baselines

Logistic regression and an RBF-kernel SVM (the classifier family of Gruss et al., 2015) are trained on the same
LOSO folds, z-scored features and feature sets as the boosted models. The reported baselines use library defaults
(C = 1, gamma = "scale"), just as the boosted models are not tuned on Part B. A small C grid is kept as a sensitivity
check: choosing its best value on the LOSO results favours the baselines, so it makes the comparison conservative.
SVM scores are a softmax of the decision values; they rank trials for AUC but are not calibrated probabilities.

In [16]:
from joblib import Parallel, delayed
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

BASELINE_GRID = {"logreg": [0.01, 0.1, 1.0], "svm": [0.1, 1.0, 10.0]}
BASELINES = ["logreg", "svm"]                                        # the C = 1 defaults
MODEL_NAMES.update({"logreg": "LogReg", "svm": "SVM-RBF"})


def baseline_key(kind, C):
    return kind if C == 1.0 else f"{kind}_C{C:g}"


BASELINE_KEYS = [baseline_key(kind, C) for kind, Cs in BASELINE_GRID.items() for C in Cs]
for kind, Cs in BASELINE_GRID.items():
    for C in Cs:
        MODEL_NAMES.setdefault(baseline_key(kind, C), f"{MODEL_NAMES[kind]} C={C:g}")


def make_baseline(kind, C):
    if kind == "logreg":
        return LogisticRegression(C=C, max_iter=5000)                 # multinomial, L2, lbfgs
    return SVC(C=C, kernel="rbf", gamma="scale")                      # libsvm, one-vs-one


def baseline_fold(X_train, y_train, X_test, n_classes, grid):
    out = {}
    for kind, Cs in BASELINE_GRID.items():
        for C in (Cs if grid else [1.0]):
            model = make_baseline(kind, C).fit(X_train, y_train)
            if kind == "logreg":
                scores = model.predict_proba(X_test)
            else:
                d = model.decision_function(X_test)
                scores = softmax(np.c_[-d, d] / 2 if d.ndim == 1 else d, axis=1)
            proba = np.zeros((len(X_test), n_classes))
            proba[:, model.classes_] = scores
            out[baseline_key(kind, C)] = proba
    return out


def run_loso_baselines(name, table, feature_cols, label_map=None, grid=False):
    """Baselines on the folds of run `name` (folds in parallel); C = 1 only unless grid=True.
    Writes loso_predictions_<name>__baselines.parquet."""
    df, X, y, labels, groups = prepare_run(table, feature_cols, label_map)
    subjects = np.unique(groups)
    keys = BASELINE_KEYS if grid else BASELINES
    t0 = time.perf_counter()
    outs = Parallel(n_jobs=N_JOBS)(
        delayed(baseline_fold)(X[groups != s], y[groups != s], X[groups == s], len(labels), grid) for s in subjects)
    blocks = []
    for s, out in zip(subjects, outs):
        test = groups == s
        for key, proba in out.items():
            block = pd.DataFrame({"model": key, "sample_id": df["sample_id"].to_numpy()[test], "subject_id": int(s),
                                  "true_class": y[test], "pred_class": proba.argmax(axis=1)})
            for j in labels:
                block[f"prob_{j}"] = proba[:, j]
            blocks.append(block)
    preds = pd.concat(blocks, ignore_index=True)
    assert len(preds) == len(keys) * len(df)
    metrics = {k: core_metrics(b["true_class"].to_numpy(), b["pred_class"].to_numpy(), labels)
               for k, b in preds.groupby("model", sort=False)}
    preds.to_parquet(MODELS_DIR / f"loso_predictions_{name}__baselines.parquet", index=False)
    write_json(MODELS_DIR / f"loso_{name}__baselines_manifest.json", {
        "run": name, "n_samples": int(len(df)), "n_subjects": int(len(subjects)), "n_features": len(feature_cols),
        "labels": labels, "label_map": label_map, "models": keys, "grid": BASELINE_GRID if grid else "C = 1 only",
        "logreg": "sklearn LogisticRegression, multinomial, L2, lbfgs, max_iter 5000",
        "svm": "sklearn SVC, RBF kernel, gamma='scale', one-vs-one; scores = softmax of the decision values",
        "seconds": time.perf_counter() - t0, "metrics": metrics,
    })
    print(f"  baselines for {name} ({time.perf_counter() - t0:.0f}s): "
          + ", ".join(f"{MODEL_NAMES[k]} {metrics[k]['accuracy']:.4f}" for k in keys))
    return metrics


model_manifest = json.loads((FEATURES_DIR / "features_model_manifest.json").read_text(encoding="utf-8"))
kept_cols = model_manifest["feature_columns"]
OURS_169 = [c for c in kept_cols if c not in FACIAL_SIM_COLS]
model_loo = pd.read_parquet(FEATURES_DIR / "features_model_partB_loo.parquet")
MAIN_RUNS = {   # leakage-controlled runs of Step 4: run -> (features, label map)
    "loo_5class_169": (OURS_169, None),
    "loo_4class_pa1ex": (kept_cols, FOUR_CLASS["label_map"]),
    "loo_5class_nofacial": (NOFACIAL_COLS, None),
    "loo_4class_nofacial": (NOFACIAL_COLS, FOUR_CLASS["label_map"]),
}

In [17]:
rows = []
for run, (cols, label_map) in MAIN_RUNS.items():
    base = run_loso_baselines(run, model_loo, cols, label_map, grid=True)
    boosted = json.loads((MODELS_DIR / f"loso_{run}_manifest.json").read_text(encoding="utf-8"))["metrics"]
    for key, c in {**boosted, **base}.items():
        rows.append({"run": run, "model": MODEL_NAMES[key], "accuracy": c["accuracy"], "macro_f1": c["macro_f1"],
                     "kappa_quadratic": c["kappa_quadratic"], "mae": c["mae"]})
print()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  baselines for loo_5class_169 (288s): LogReg C=0.01 0.3401, LogReg C=0.1 0.3380, LogReg 0.3373, SVM-RBF C=0.1 0.3295, SVM-RBF 0.3404, SVM-RBF C=10 0.3200


  baselines for loo_4class_pa1ex (185s): LogReg C=0.01 0.4176, LogReg C=0.1 0.4213, LogReg 0.4197, SVM-RBF C=0.1 0.4145, SVM-RBF 0.4206, SVM-RBF C=10 0.4073


  baselines for loo_5class_nofacial (131s): LogReg C=0.01 0.3172, LogReg C=0.1 0.3171, LogReg 0.3170, SVM-RBF C=0.1 0.3120, SVM-RBF 0.3160, SVM-RBF C=10 0.2930


  baselines for loo_4class_nofacial (78s): LogReg C=0.01 0.3908, LogReg C=0.1 0.3887, LogReg 0.3889, SVM-RBF C=0.1 0.3913, SVM-RBF 0.3926, SVM-RBF C=10 0.3732

                run         model  accuracy  macro_f1  kappa_quadratic    mae
     loo_5class_169       XGBoost    0.3604    0.3541           0.4513 1.0701
     loo_5class_169           HGB    0.3509    0.3465           0.4307 1.0910
     loo_5class_169 LogReg C=0.01    0.3401    0.3317           0.4328 1.1268
     loo_5class_169  LogReg C=0.1    0.3380    0.3319           0.4254 1.1264
     loo_5class_169        LogReg    0.3373    0.3320           0.4219 1.1271
     loo_5class_169 SVM-RBF C=0.1    0.3295    0.3190           0.3930 1.1449
     loo_5class_169       SVM-RBF    0.3404    0.3367           0.4314 1.1030
     loo_5class_169  SVM-RBF C=10    0.3200    0.3176           0.3794 1.1596
   loo_4class_pa1ex       XGBoost    0.4401    0.4368           0.4867 0.7825
   loo_4class_pa1ex           HGB    0.4346    0.4320       

## Step 9 — the published feature set of Gruss et al. (2015)

Gruss et al. released 159 features for 85 Part B subjects × 100 trials: 39 per EMG channel and for EDA (amplitude,
frequency, stationarity, entropy, linearity, variability, baseline similarity) and 3 for ECG. Each of their rows is
linked to one released trial by its position in time: the release numbers pain trials 1–80 and BL1 trials 81–100 in
time order, and the published rows carry the window start. The link is verified against quantities both pipelines
measure. Only trials retained in Step 1 are used, so both feature sets are evaluated on identical trials, subjects,
folds, normalisation and models. Missing published values are filled with the subject's median.

The published features were computed from the original recordings, so they also show which class of the release
holds the original facial EMG: per class, our facial-EMG amplitude is compared with the published one. The audit also
checks every similarity feature for self-inclusion (a BL1 trial compared with a template that contains it).

Every task is run in three variants, on identical trials:

| variant | ours | published | purpose |
|---|---|---|---|
| facial EMG kept | 169 (175 for 4-class) | 147 (159 for 4-class) | each set as designed, minus the leaking facial similarity |
| `nofacial` | no corrugator/zygomaticus | no CH22/CH23 | only channels intact in every class of the release |
| `nofacial_nosim` | as above, no similarity features | as above, no Sim- features | strict like-for-like: no template-based feature on either side |

Gruss et al. reported SVM accuracies from a random 75/25 trial split, so every subject was in both training and test
set, after forward feature selection. The last cell of this step repeats a random split to separate the effect of
the protocol from the effect of the features. The step is skipped when `BIOVID_PLOS_FEATURES` is not set.

In [18]:
PLOS_CHANNELS = {"CH22": "emg_zygomaticus", "CH23": "emg_corrugator", "CH24": "emg_trapezius", "CH25": "ecg", "CH26": "gsr"}
PLOS_CLASSES = {"level_zero_one": "BL1", "level_one": "PA1", "level_two": "PA2", "level_three": "PA3", "level_four": "PA4"}
HAVE_PLOS = PLOS_FEATURES_FILE is not None


def plos_subject_name(filename):
    """'20120713-09_w_21' -> '071309_w_21', the folder name in the release."""
    m = re.match(r"^\d{4}(\d{2})(\d{2})-(\d{2})_(.+)$", filename)
    return f"{m.group(1)}{m.group(2)}{m.group(3)}_{m.group(4)}"


def within_subject_r(df, a, b):
    z = df.groupby("subject_id")[[a, b]].transform(lambda s: (s - s.mean()) / (s.std() + 1e-12))
    return float(np.corrcoef(z[a], z[b])[0, 1])


def load_published_features():
    t0 = time.perf_counter()
    sheet = pd.read_excel(PLOS_FEATURES_FILE, sheet_name="ALL_CH_&_FEATS")
    src = [c for c in sheet.columns if re.match(r"^CH2[2-6]_", c)]
    assert len(src) == 159, len(src)
    cols = [f"plos_{PLOS_CHANNELS[c[:4]]}_{c[5:]}" for c in src]
    values = sheet[src].apply(pd.to_numeric, errors="coerce").astype(np.float64)   # some L-pldf cells exceed int64
    values.columns = cols
    plos = pd.DataFrame({"subject_name": sheet["Filename"].map(plos_subject_name),
                         "class_name": sheet["Label"].map(PLOS_CLASSES),
                         "window_start_s": sheet["Start_Wnd (Sec)"].astype(float)}).join(values)
    assert plos["class_name"].notna().all() and plos.groupby("subject_name").size().eq(100).all()
    print(f"read {len(plos)} rows x {len(cols)} features from {plos['subject_name'].nunique()} subjects "
          f"in {time.perf_counter() - t0:.0f}s")

    # ---- link each row to a released trial: chronological rank within subject and class ----
    plos = plos.sort_values(["subject_name", "window_start_s"]).reset_index(drop=True)
    plos["rank"] = plos.groupby(["subject_name", "class_name"]).cumcount()
    index = pd.read_parquet(FEATURES_DIR / "index_biosignals_partB.parquet").sort_values(["subject_name", "sample_num"])
    index["rank"] = index.groupby(["subject_name", "class_name"]).cumcount()
    common = sorted(set(plos["subject_name"]) & set(index["subject_name"]))
    for s in common:                                                 # identical pain-level sequence in both sources
        a = plos.loc[(plos["subject_name"] == s) & (plos["class_name"] != "BL1"), "class_name"].tolist()
        b = index.loc[(index["subject_name"] == s) & (index["class_name"] != "BL1"), "class_name"].tolist()
        assert a == b, f"pain sequence differs for {s}"
    linked = plos[plos["subject_name"].isin(common)].merge(
        index[["subject_name", "class_name", "rank", "sample_name"]], on=["subject_name", "class_name", "rank"],
        how="inner", validate="one_to_one")
    assert len(linked) == 100 * len(common)

    n_nan = linked[cols].isna().sum()
    linked[cols] = linked[cols].fillna(linked.groupby("subject_name")[cols].transform("median"))
    assert linked[cols].notna().all().all()

    meta = pd.read_parquet(CACHE_DIR / "meta.parquet")
    plos_matched = (meta[ID_COLS].merge(linked[["sample_name"] + cols], on="sample_name", how="inner", validate="one_to_one")
                    .sort_values("sample_id").reset_index(drop=True))
    ours = pd.read_parquet(FEATURES_DIR / "features_model_partB_loo.parquet")
    ours_matched = ours[ours["sample_id"].isin(plos_matched["sample_id"])].sort_values("sample_id").reset_index(drop=True)
    assert ours_matched["sample_id"].equals(plos_matched["sample_id"])

    # ---- verify the link: within-subject r of quantities both pipelines measure; a one-trial shift is the control ----
    raw = pd.read_parquet(FEATURES_DIR / "features_raw_partB.parquet")
    pairs = [("ecg_rr_mean_s", "plos_ecg_meanRR"), ("gsr_raw_mean", "plos_gsr_A-MAV"),
             ("emg_trapezius_rms", "plos_emg_trapezius_A-RMS"), ("emg_corrugator_rms", "plos_emg_corrugator_A-RMS"),
             ("emg_zygomaticus_rms", "plos_emg_zygomaticus_A-RMS")]
    j = plos_matched[["sample_id", "subject_id", "class_id"] + [b for _, b in pairs]].merge(
        raw[["sample_id"] + [a for a, _ in pairs]], on="sample_id").sort_values("sample_id")
    bl1 = j["class_id"] == 0
    link_check = []
    for a, b in pairs:
        shifted = j.groupby(["subject_id", "class_id"])[b].transform(lambda s: np.roll(s.to_numpy(), 1))
        link_check.append({"ours": a, "published": b, "r_bl1": within_subject_r(j[bl1], a, b),
                           "r_pain": within_subject_r(j[~bl1], a, b),
                           "r_shifted_by_one_trial": within_subject_r(j.assign(**{b: shifted}), a, b)})
    link_check = pd.DataFrame(link_check)
    intact = ~link_check["ours"].str.contains("corrugator|zygomaticus")   # facial BL1 is examined in the audit cell
    assert (link_check.loc[intact, ["r_bl1", "r_pain"]] > 0.5).all().all(), link_check
    assert (link_check["r_pain"] > 0.5).all(), link_check

    info = {
        "source_file": PLOS_FEATURES_FILE.name, "n_features": len(cols),
        "subjects_published": int(plos["subject_name"].nunique()), "subjects_common": len(common),
        "subjects_only_published": sorted(set(plos["subject_name"]) - set(common)),
        "subjects_only_ours": sorted(set(index["subject_name"]) - set(common)),
        "trials_linked": int(len(linked)), "trials_matched": int(len(plos_matched)),
        "trials_dropped_by_our_quality_filter": int(len(linked) - len(plos_matched)),
        "class_counts_matched": {CLASS_NAMES[int(k)]: int(v) for k, v in plos_matched["class_id"].value_counts().sort_index().items()},
        "nan_cells_imputed": {c: int(v) for c, v in n_nan.items() if v > 0},
        "imputation": "within-subject median over the subject's 100 published rows",
        "link": "chronological rank within subject and class (pain trials 1-80, BL1 81-100 in the release)",
        "link_check": link_check.to_dict("records"),
    }
    plos_matched.to_parquet(FEATURES_DIR / "features_plos_matched.parquet", index=False)
    write_json(FEATURES_DIR / "features_plos_manifest.json", {**info, "feature_columns": cols,
                                                              "channel_map": PLOS_CHANNELS, "class_map": PLOS_CLASSES})
    print(f"subjects: {len(common)} common, only published {info['subjects_only_published']}, only ours {info['subjects_only_ours']}")
    print(f"trials  : {len(linked)} linked, {len(plos_matched)} kept after our quality filter")
    print("imputed NaN cells:", info["nan_cells_imputed"])
    print("\nlink check (within-subject Pearson r):")
    print(link_check.round(3).to_string(index=False))
    return plos_matched, ours_matched, linked, cols, info


if HAVE_PLOS:
    plos_matched, ours_matched, plos_linked, PLOS_ALL, plos_info = load_published_features()
    PLOS_FACIAL_SIM = [c for c in PLOS_ALL if re.match(r"^plos_emg_(corrugator|zygomaticus)_Sim-", c)]
    PLOS_NOFAC = [c for c in PLOS_ALL if c not in PLOS_FACIAL_SIM]
    PLOS_NOFACIAL = [c for c in PLOS_ALL if not c.startswith(("plos_emg_corrugator_", "plos_emg_zygomaticus_"))]
    PLOS_NOFACIAL_NOSIM = [c for c in PLOS_NOFACIAL if "_Sim-" not in c]
    OURS_NOFACIAL_NOSIM = [c for c in NOFACIAL_COLS if "_sim_" not in c]
    print(f"\npublished features: {len(PLOS_ALL)} all, {len(PLOS_NOFAC)} without the {len(PLOS_FACIAL_SIM)} facial "
          f"similarity features, {len(PLOS_NOFACIAL)} without facial EMG, {len(PLOS_NOFACIAL_NOSIM)} also without similarity")
    print(f"our features      : {len(kept_cols)} all, {len(OURS_169)} without facial similarity, {len(NOFACIAL_COLS)} without "
          f"facial EMG, {len(OURS_NOFACIAL_NOSIM)} also without similarity")
else:
    print("BIOVID_PLOS_FEATURES is not set: Step 9 is skipped.")

read 8500 rows x 159 features from 85 subjects in 5s


subjects: 84 common, only published ['071313_m_41'], only ours ['092014_m_56', '120810_m_38']
trials  : 8400 linked, 8226 kept after our quality filter
imputed NaN cells: {'plos_emg_zygomaticus_L-pldf': 5, 'plos_emg_corrugator_L-pldf': 9, 'plos_emg_trapezius_L-pldf': 8, 'plos_gsr_S-areapond': 48, 'plos_gsr_L-pldf': 3}

link check (within-subject Pearson r):
               ours                  published  r_bl1  r_pain  r_shifted_by_one_trial
      ecg_rr_mean_s            plos_ecg_meanRR  0.914   0.896                   0.164
       gsr_raw_mean             plos_gsr_A-MAV  0.804   0.794                   0.361
  emg_trapezius_rms   plos_emg_trapezius_A-RMS  0.802   0.887                   0.220
 emg_corrugator_rms  plos_emg_corrugator_A-RMS  0.341   0.957                   0.322
emg_zygomaticus_rms plos_emg_zygomaticus_A-RMS  0.222   0.967                   0.227

published features: 159 all, 147 without the 12 facial similarity features, 81 without facial EMG, 69 also without similari

In [19]:
def audit_published_features():
    """Does the published table carry the PA1 = BL1 facial copy of the release, and do its facial similarity features
    separate BL1 from PA1 the way ours do?"""
    meta = pd.read_parquet(CACHE_DIR / "meta.parquet")
    twins = pd.read_parquet(REPORTS_DIR / "pa1_twin_pairs.parquet")
    name_of = meta.set_index("sample_id")["sample_name"]
    by_name = plos_linked.set_index("sample_name")
    twins = twins[name_of[twins["sample_id_bl1"]].isin(by_name.index).to_numpy()]      # subjects in both sources
    raw =pd.read_parquet(FEATURES_DIR / "features_raw_partB.parquet").set_index("sample_name")
    dup_rows = []
    for ch in ("corrugator", "zygomaticus", "trapezius"):
        cols = [c for c in PLOS_ALL if c.startswith(f"plos_emg_{ch}_") and "_Sim-" not in c]
        matched = total = 0
        for _, g in plos_linked.groupby("subject_name"):
            bl1 = {r.tobytes() for r in g.loc[g["class_name"] == "BL1", cols].to_numpy()}
            pa1 = [r.tobytes() for r in g.loc[g["class_name"] == "PA1", cols].to_numpy()]
            matched += sum(r in bl1 for r in pa1)
            total += len(pa1)
        bl1_names = name_of[twins["sample_id_bl1"]].to_numpy()
        pa1_names = name_of[twins["sample_id_pa1"]].to_numpy()
        tw = pd.DataFrame({"subject_id": twins["subject_id"].to_numpy(),
                           "ours_bl1": raw.loc[bl1_names, f"emg_{ch}_rms"].to_numpy(),
                           "ours_pa1": raw.loc[pa1_names, f"emg_{ch}_rms"].to_numpy(),
                           "pub_bl1": by_name.loc[bl1_names, f"plos_emg_{ch}_A-RMS"].to_numpy(),
                           "pub_pa1": by_name.loc[pa1_names, f"plos_emg_{ch}_A-RMS"].to_numpy()})
        dup_rows.append({"channel": ch, "published_non_similarity_features": len(cols),
                         "published_pa1_rows_identical_to_a_bl1_row": int(matched), "published_pa1_rows": int(total),
                         "twin_pairs": int(len(twins)),
                         "twin_r_ours_rms": within_subject_r(tw, "ours_bl1", "ours_pa1"),
                         "twin_r_published_rms": within_subject_r(tw, "pub_bl1", "pub_pa1")})
    dup = pd.DataFrame(dup_rows)

    # which class of the release holds the original recording? within-subject r of our vs the published amplitude
    j = plos_matched.merge(raw.reset_index()[["sample_id"] + [f"emg_{ch}_rms" for ch in ("corrugator", "zygomaticus", "trapezius")]],
                           on="sample_id")
    direction = pd.DataFrame([{"channel": ch, **{CLASS_NAMES[k]: within_subject_r(j[j["class_id"] == k], f"emg_{ch}_rms",
                                                                                f"plos_emg_{ch}_A-RMS") for k in LABELS}}
                              for ch in ("corrugator", "zygomaticus", "trapezius")])

    # single-feature BL1-vs-PA1 AUC of the facial similarity features (within-subject z-scored)
    auc_rows = []
    for label, table, cols in [("ours (leave-one-out template)", ours_matched, [c for c in FACIAL_SIM_COLS]),
                               ("published", plos_matched, PLOS_FACIAL_SIM)]:
        sub = table[table["class_id"].isin([0, 1])]
        Z, _ = zscore_within_subject(sub[cols].to_numpy(), sub["subject_id"].to_numpy())
        y01 = (sub["class_id"].to_numpy() == 1).astype(int)
        for k, c in enumerate(cols):
            auc = float(roc_auc_score(y01, Z[:, k]))
            auc_rows.append({"feature_set": label, "feature": c, "auc_pa1_vs_bl1": auc, "separation": abs(auc - 0.5) * 2})
    auc = pd.DataFrame(auc_rows)

    # self-inclusion check of every similarity feature: BL1 vs PA2-PA4 (PA1 left out because of the facial copy).
    # A template that contains the trial itself makes BL1 look similar to its template (AUC far from 0.5) even on
    # channels whose leave-one-out version carries no information.
    feats = pd.read_parquet(FEATURES_DIR / "features_raw_partB.parquet").set_index("sample_id").loc[plos_matched["sample_id"]]
    keep = plos_matched["class_id"].to_numpy() != 1
    y_bl1 = (plos_matched["class_id"].to_numpy()[keep] == 0).astype(int)
    subj = plos_matched["subject_id"].to_numpy()

    def bl1_auc(values):
        z, _ = zscore_within_subject(np.asarray(values, dtype=np.float64)[:, None], subj)
        return float(roc_auc_score(y_bl1, z[keep, 0]))

    sim_rows = []
    for c in [c for c in PLOS_ALL if "_Sim-" in c]:
        sim_rows.append({"feature_set": "published", "feature": c, "auc_bl1_vs_pa2_pa4": bl1_auc(plos_matched[c])})
    for c in [c for c in feats.columns if "_sim_" in c and not c.startswith(LOO_PREFIX)]:
        sim_rows.append({"feature_set": "ours, full template", "feature": c, "auc_bl1_vs_pa2_pa4": bl1_auc(feats[c])})
        sim_rows.append({"feature_set": "ours, leave-one-out", "feature": c, "auc_bl1_vs_pa2_pa4": bl1_auc(feats[LOO_PREFIX + c])})
    sim = pd.DataFrame(sim_rows)
    sim["separation"] = (sim["auc_bl1_vs_pa2_pa4"] - 0.5).abs() * 2
    sim["channel"] = sim["feature"].map(lambda c: next(ch for ch in ("gsr", "ecg", "trapezius", "corrugator", "zygomaticus") if ch in c))
    sim_summary = sim.pivot_table(index="channel", columns="feature_set", values="separation", aggfunc="max")

    print("PA1 rows of the published table that equal a BL1 row of the same subject (non-similarity EMG features),")
    print("and the correlation of released twin signals with the published BL1 and PA1 rows:")
    print(dup.round(3).to_string(index=False))
    print("\nour vs published EMG amplitude per class (within-subject r); a low value marks a class whose released")
    print("signal is not the recording the published features were computed from:")
    print(direction.round(3).to_string(index=False))
    print("\nBL1-vs-PA1 AUC of single facial similarity features (0.5 = no information):")
    print(auc.round(3).to_string(index=False))
    facial = direction[direction["channel"] != "trapezius"]
    copied_class = "BL1" if (facial["BL1"] < 0.6).all() and (facial["PA1"] > 0.8).all() else (
        "PA1" if (facial["PA1"] < 0.6).all() and (facial["BL1"] > 0.8).all() else "undetermined")
    print(f"\nreleased class whose facial EMG is a copy of the other class: {copied_class}")
    print("\nself-inclusion check: largest BL1-vs-PA2..PA4 separation |2 AUC - 1| of a single similarity feature")
    print(sim_summary.round(3).to_string())
    audit = {"duplication": dup.to_dict("records"), "per_class_agreement": direction.to_dict("records"),
             "copied_class": copied_class, "facial_similarity_auc": auc.to_dict("records"),
             "max_separation": {k: float(v) for k, v in auc.groupby("feature_set")["separation"].max().items()},
             "similarity_self_inclusion": sim.to_dict("records"),
             "similarity_self_inclusion_max_separation": {ch: {k: float(v) for k, v in row.items()}
                                                          for ch, row in sim_summary.iterrows()}}
    write_json(REPORTS_DIR / "published_features_audit.json", audit)
    return audit


if HAVE_PLOS:
    plos_audit = audit_published_features()

PA1 rows of the published table that equal a BL1 row of the same subject (non-similarity EMG features),
and the correlation of released twin signals with the published BL1 and PA1 rows:
    channel  published_non_similarity_features  published_pa1_rows_identical_to_a_bl1_row  published_pa1_rows  twin_pairs  twin_r_ours_rms  twin_r_published_rms
 corrugator                                 33                                         14                1680        1616            1.000                 0.308
zygomaticus                                 33                                         16                1680        1616            1.000                 0.212
  trapezius                                 33                                          0                1680        1616            0.424                 0.324

our vs published EMG amplitude per class (within-subject r); a low value marks a class whose released
signal is not the recording the published features were computed fr

In [20]:
COMPARISON_TASKS = {}
if HAVE_PLOS:
    FOUR = FOUR_CLASS["label_map"]
    VARIANTS = {   # suffix -> (ours, published, label) for 5-class/binary tasks; 4-class keeps facial similarity
        "": (OURS_169, PLOS_NOFAC, "facial EMG without facial similarity"),
        "_nofacial": (NOFACIAL_COLS, PLOS_NOFACIAL, "no facial EMG"),
        "_nofacial_nosim": (OURS_NOFACIAL_NOSIM, PLOS_NOFACIAL_NOSIM, "no facial EMG, no similarity features"),
    }
    COMPARISON_TASKS = {}
    for v, (ours_cols, plos_cols, text) in VARIANTS.items():
        COMPARISON_TASKS[f"5class{v}"] = dict(label_map=None, ours=ours_cols, plos=plos_cols, label=f"5-class, {text}")
        four = (kept_cols, PLOS_ALL, "all features") if v == "" else (ours_cols, plos_cols, text)
        COMPARISON_TASKS["4class_pa1ex" if v == "" else f"4class{v}"] = dict(
            label_map=FOUR, ours=four[0], plos=four[1], label=f"4-class (BL1, PA2-PA4), {four[2]}")
        for k in (1, 2, 3, 4):
            COMPARISON_TASKS[f"bl1_vs_pa{k}{v}"] = dict(label_map={0: 0, k: 1}, ours=ours_cols, plos=plos_cols,
                                                        label=f"BL1 vs PA{k}, {text}")


def run_comparison(task):
    spec = COMPARISON_TASKS[task]
    for fs, table in (("ours", ours_matched), ("plos", plos_matched)):
        source = "our" if fs == "ours" else "published (Gruss et al. 2015)"
        run_loso(f"cmp_{task}_{fs}", table, spec[fs], label_map=spec["label_map"],
                 note=f"{spec['label']}: {source} features ({len(spec[fs])}) on the matched trials")
        run_loso_baselines(f"cmp_{task}_{fs}", table, spec[fs], spec["label_map"])


for task in [t for t in COMPARISON_TASKS if not t.startswith("bl1_vs")]:
    run_comparison(task)

cmp_5class_ours: 8226 trials, 84 subjects, 169 features, 5 classes


  fold 10/84      25s  eta  3.1 min


  fold 20/84      51s  eta  2.7 min


  fold 30/84      77s  eta  2.3 min


  fold 40/84     103s  eta  1.9 min


  fold 50/84     128s  eta  1.5 min


... (111 outputs left out)

  fold 70/84      84s  eta  0.3 min


  fold 80/84      95s  eta  0.1 min


  fold 84/84     100s  eta  0.0 min
  XGBoost  acc 0.3548  macro-F1 0.3503  kappa_q 0.3152  MAE 0.974  fold acc median 0.325
  HGB      acc 0.3553  macro-F1 0.3525  kappa_q 0.3116  MAE 0.970  fold acc median 0.325
  top-5 XGB gain: plos_gsr_S-sd 0.040, plos_gsr_F-fmode 0.040, plos_gsr_F-fmean 0.031, plos_gsr_A-P2P 0.028, plos_gsr_F-cf 0.025
  100s total, folds where xgb>hgb/hgb>xgb/tie: 37/36/11


  baselines for cmp_4class_nofacial_nosim_plos (14s): LogReg 0.3554, SVM-RBF 0.3480


In [21]:
for task in [t for t in COMPARISON_TASKS if t.startswith("bl1_vs")]:
    run_comparison(task)

cmp_bl1_vs_pa1_ours: 3259 trials, 84 subjects, 169 features, 2 classes


  fold 10/84       4s  eta  0.5 min


  fold 20/84       9s  eta  0.5 min


  fold 30/84      13s  eta  0.4 min


  fold 40/84      17s  eta  0.3 min


  fold 50/84      21s  eta  0.2 min


... (231 outputs left out)

  fold 70/84      19s  eta  0.1 min


  fold 80/84      22s  eta  0.0 min


  fold 84/84      23s  eta  0.0 min
  XGBoost  acc 0.7078  macro-F1 0.7077  kappa_q 0.4154  MAE 0.292  fold acc median 0.703
  HGB      acc 0.7072  macro-F1 0.7071  kappa_q 0.4142  MAE 0.293  fold acc median 0.700
  top-5 XGB gain: plos_gsr_S-sd 0.071, plos_gsr_F-fmode 0.048, plos_emg_trapezius_A-MAVFD 0.031, plos_gsr_F-fmean 0.031, plos_ecg_meanRR 0.030
  23s total, folds where xgb>hgb/hgb>xgb/tie: 27/30/27


  baselines for cmp_bl1_vs_pa4_nofacial_nosim_plos (2s): LogReg 0.6969, SVM-RBF 0.6838


In [22]:
from sklearn.model_selection import train_test_split

# Accuracies reported by Gruss et al. (2015) with a random 75/25 split and forward feature selection (RBF-SVM).
PUBLISHED_ACCURACY = {"bl1_vs_pa1": 0.7929, "bl1_vs_pa3": 0.8494, "bl1_vs_pa4": 0.9094}
SPLIT_REPEATS = 5
SPLIT_TASKS = ["5class", "bl1_vs_pa1", "bl1_vs_pa3", "bl1_vs_pa4", "bl1_vs_pa4_nofacial", "bl1_vs_pa4_nofacial_nosim"]


def random_split_accuracy(table, cols, label_map):
    """Mean accuracy over repeated stratified 75/25 trial splits: every subject is in training and test data."""
    df, X, y, labels, groups = prepare_run(table, cols, label_map)
    rows = []
    for r in range(SPLIT_REPEATS):
        tr, te = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED + r)
        models = {"xgb": XGBClassifier(**(XGB_PARAMS if len(labels) > 2 else XGB_BINARY_PARAMS)),
                  "hgb": HistGradientBoostingClassifier(**HGB_PARAMS),
                  "logreg": make_baseline("logreg", 1.0), "svm": make_baseline("svm", 1.0)}
        for m, model in models.items():
            rows.append({"model": m, "repeat": r, "accuracy": float(accuracy_score(y[te], model.fit(X[tr], y[tr]).predict(X[te])))})
    return pd.DataFrame(rows).groupby("model")["accuracy"].agg(["mean", "std"])


def protocol_comparison():
    t0 = time.perf_counter()
    rows = []
    for task in SPLIT_TASKS:
        spec = COMPARISON_TASKS[task]
        for fs, table in (("ours", ours_matched), ("plos", plos_matched)):
            split = random_split_accuracy(table, spec[fs], spec["label_map"])
            loso = json.loads((MODELS_DIR / f"loso_cmp_{task}_{fs}_manifest.json").read_text(encoding="utf-8"))["metrics"]
            loso.update(json.loads((MODELS_DIR / f"loso_cmp_{task}_{fs}__baselines_manifest.json").read_text(encoding="utf-8"))["metrics"])
            for m in ["xgb", "hgb", "logreg", "svm"]:
                rows.append({"task": task, "features": fs, "model": m, "random_split_acc": split.loc[m, "mean"],
                             "random_split_sd": split.loc[m, "std"], "loso_acc": loso[m]["accuracy"],
                             "published_acc": PUBLISHED_ACCURACY.get(task, np.nan)})
    table = pd.DataFrame(rows)
    table["split_minus_loso"] = table["random_split_acc"] - table["loso_acc"]
    print(f"random-split protocol, {SPLIT_REPEATS} repeats ({time.perf_counter() - t0:.0f}s)")
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    write_json(REPORTS_DIR / "protocol_random_split.json", {
        "protocol": f"stratified random 75/25 trial split, {SPLIT_REPEATS} repeats, within-subject z-score, no feature selection",
        "repeats": SPLIT_REPEATS,
        "published_accuracy_gruss_2015": PUBLISHED_ACCURACY,
        "published_protocol": "RBF-SVM with grid search and forward feature selection on a random 75/25 split",
        "rows": table.to_dict("records")})
    return table


if HAVE_PLOS:
    protocol_table = protocol_comparison()

random-split protocol, 5 repeats (83s)
                     task features  model  random_split_acc  random_split_sd  loso_acc  published_acc  split_minus_loso
                   5class     ours    xgb            0.3544           0.0072    0.3549            NaN           -0.0005
                   5class     ours    hgb            0.3492           0.0082    0.3530            NaN           -0.0038
                   5class     ours logreg            0.3368           0.0140    0.3364            NaN            0.0004
                   5class     ours    svm            0.3324           0.0077    0.3369            NaN           -0.0044
                   5class     plos    xgb            0.3921           0.0064    0.3906            NaN            0.0015
                   5class     plos    hgb            0.3905           0.0111    0.3848            NaN            0.0057
                   5class     plos logreg            0.3809           0.0069    0.3762            NaN            0.0047
 

## Step 10 — comparisons with paired subject-level bootstraps

Every comparison is paired: both arms are scored on the same trials, and each bootstrap resample draws subjects
(2000 resamples, the same draws as Step 5). The model comparison tests the boosted models against the baselines on
each leakage-controlled run. The feature-set comparison tests our features against the published ones for each model
and task.

In [23]:
EVAL_MODELS = ["xgb", "hgb", "logreg", "svm"]
SCORE_KEYS = ["accuracy", "macro_f1", "kappa_quadratic"]
CHANCE = {5: 0.20, 4: 0.25, 2: 0.50}


def md_table(df, fmt="{:.4f}"):
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "|".join("---" for _ in df.columns) + "|"]
    for row in df.itertuples(index=False):
        lines.append("| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row) + " |")
    return "\n".join(lines)


def load_all_predictions(run):
    frames = [pd.read_parquet(MODELS_DIR / f"loso_predictions_{run}.parquet")]
    if (MODELS_DIR / f"loso_predictions_{run}__baselines.parquet").exists():
        frames.append(pd.read_parquet(MODELS_DIR / f"loso_predictions_{run}__baselines.parquet"))
    preds = pd.concat(frames, ignore_index=True)
    labels = sorted(int(c[len("prob_"):]) for c in preds.columns if c.startswith("prob_"))
    blocks = {m: g.sort_values("sample_id").reset_index(drop=True) for m, g in preds.groupby("model", sort=False)}
    ref = blocks["xgb"]["sample_id"].to_numpy()
    assert all(np.array_equal(b["sample_id"].to_numpy(), ref) for b in blocks.values())
    return blocks, labels


def fast_scores(y, p, k):
    """Accuracy, macro-F1 and quadratic kappa from one confusion matrix (all classes present in y)."""
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k).astype(float)
    n, tp, rows, cols = cm.sum(), np.diag(cm), cm.sum(axis=1), cm.sum(axis=0)
    f1 = np.divide(2 * tp, rows + cols, out=np.zeros(k), where=(rows + cols) > 0)
    w = np.subtract.outer(np.arange(k), np.arange(k)) ** 2 / (k - 1) ** 2
    expected = np.outer(rows, cols) / n
    return tp.sum() / n, f1.mean(), 1.0 - (w * cm).sum() / (w * expected).sum()


def bootstrap_scores(subject, y, preds, k, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    uniq = np.unique(subject)
    idx_map = {s: np.flatnonzero(subject == s) for s in uniq}
    draws = {m: np.empty((n_boot, len(SCORE_KEYS))) for m in preds}
    for b in range(n_boot):
        idx = np.concatenate([idx_map[s] for s in rng.choice(uniq, size=len(uniq), replace=True)])
        for m, p in preds.items():
            draws[m][b] = fast_scores(y[idx], p[idx], k)
    return draws


def paired_delta(draws, point, a, b):
    out = {}
    for i, key in enumerate(SCORE_KEYS):
        d = draws[a][:, i] - draws[b][:, i]
        out[key] = {"delta": point[a][key] - point[b][key], "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
                    "p_two_sided": float(min(1.0, 2 * min(np.mean(d > 0), np.mean(d < 0))))}
    return out


def auc_score(y, prob, labels):
    prob = prob / prob.sum(axis=1, keepdims=True)
    if len(labels) == 2:
        return float(roc_auc_score(y, prob[:, 1]))
    return float(roc_auc_score(y, prob, multi_class="ovr", labels=labels))


def score_block(blocks, labels, rename=lambda m: m):
    y = blocks["xgb"]["true_class"].to_numpy(dtype=int)
    subject = blocks["xgb"]["subject_id"].to_numpy(dtype=int)
    preds = {rename(m): b["pred_class"].to_numpy(dtype=int) for m, b in blocks.items()}
    point, auc = {}, {}
    for m, b in blocks.items():
        acc, f1, kq = fast_scores(y, preds[rename(m)], len(labels))
        point[rename(m)] = {"accuracy": acc, "macro_f1": f1, "kappa_quadratic": kq}
        auc[rename(m)] = auc_score(y, b[[f"prob_{j}" for j in labels]].to_numpy(dtype=np.float64), labels)
    ref = core_metrics(y, preds[rename("xgb")], labels)                       # fast_scores == sklearn
    assert all(abs(ref[k] - point[rename("xgb")][k]) < 1e-9 for k in SCORE_KEYS)
    return y, subject, preds, point, auc


def ci_of(draws, m):
    return {k: [float(np.percentile(draws[m][:, i], 2.5)), float(np.percentile(draws[m][:, i], 97.5))]
            for i, k in enumerate(SCORE_KEYS)}


# ---------------- boosted models vs baselines ----------------
def evaluate_models(run):
    blocks, labels = load_all_predictions(run)
    y, subject, preds, point, auc = score_block(blocks, labels)
    draws = bootstrap_scores(subject, y, preds, len(labels))
    best = {kind: max([k for k in preds if k.split("_C")[0] == kind], key=lambda k: point[k]["accuracy"])
            for kind in BASELINES}
    comparisons = [("xgb", "hgb")] + [(a, b) for a in ("xgb", "hgb") for b in BASELINES]
    comparisons += [(a, best[kind]) for a in ("xgb", "hgb") for kind in BASELINES if best[kind] != kind]
    return {"run": run, "n_samples": int(len(y)), "n_subjects": int(len(np.unique(subject))), "n_classes": len(labels),
            "models": {m: {"name": MODEL_NAMES[m], **point[m], "auc": auc[m], "ci95": ci_of(draws, m)} for m in preds},
            "best_grid_baseline": best,
            "paired": {f"{a} - {b}": paired_delta(draws, point, a, b) for a, b in comparisons}}


MODEL_RUNS = list(MAIN_RUNS) + [f"cmp_{t}_{fs}" for t in COMPARISON_TASKS for fs in ("ours", "plos")]
model_cmp = {run: evaluate_models(run) for run in MODEL_RUNS}

md = ["# Boosted models vs linear baselines\n",
      "LOSO, identical folds and features per run; 95% CIs and paired deltas from 2000 subject-level bootstrap resamples. "
      "Baselines use C = 1; *best grid* is the best C chosen on these same LOSO results (optimistic for the baseline).\n",
      "| run | trials | model | accuracy | 95% CI | macro-F1 | kappa (quadratic) | AUC |", "|---|---:|---|---:|---|---:|---:|---:|"]
for run, r in model_cmp.items():
    for m in EVAL_MODELS + [r["best_grid_baseline"][k] for k in BASELINES if r["best_grid_baseline"][k] != k]:
        v = r["models"][m]
        name = v["name"] + (" (best grid)" if m not in EVAL_MODELS else "")
        md.append(f"| `{run}` | {r['n_samples']} | {name} | {v['accuracy']:.4f} | [{v['ci95']['accuracy'][0]:.3f}, "
                  f"{v['ci95']['accuracy'][1]:.3f}] | {v['macro_f1']:.4f} | {v['kappa_quadratic']:.4f} | {v['auc']:.4f} |")
md += ["\n## Paired accuracy differences\n", "| run | comparison | delta accuracy | 95% CI | p | delta macro-F1 | 95% CI | p |",
       "|---|---|---:|---|---:|---:|---|---:|"]
for run, r in model_cmp.items():
    for pair, d in r["paired"].items():
        a, b = pair.split(" - ")
        label = f"{MODEL_NAMES[a]} - {MODEL_NAMES[b]}"
        md.append(f"| `{run}` | {label} | {d['accuracy']['delta']:+.4f} | [{d['accuracy']['ci95'][0]:+.3f}, {d['accuracy']['ci95'][1]:+.3f}] | "
                  f"{d['accuracy']['p_two_sided']:.3f} | {d['macro_f1']['delta']:+.4f} | [{d['macro_f1']['ci95'][0]:+.3f}, "
                  f"{d['macro_f1']['ci95'][1]:+.3f}] | {d['macro_f1']['p_two_sided']:.3f} |")
grid_rows = [{"run": run, **{MODEL_NAMES[k]: r["models"][k]["accuracy"] for k in BASELINE_KEYS}} for run, r in model_cmp.items()
             if all(k in r["models"] for k in BASELINE_KEYS)]
md += ["\n## Baseline accuracy across the C grid\n", md_table(pd.DataFrame(grid_rows))]
(REPORTS_DIR / "model_comparison.md").write_text("\n".join(md), encoding="utf-8")
write_json(REPORTS_DIR / "model_comparison.json", model_cmp)

view = pd.DataFrame([{"run": run, "model": r["models"][m]["name"], "accuracy": r["models"][m]["accuracy"],
                      "ci95": "[{:.3f}, {:.3f}]".format(*r["models"][m]["ci95"]["accuracy"]),
                      "macro_f1": r["models"][m]["macro_f1"], "auc": r["models"][m]["auc"]}
                     for run, r in model_cmp.items() for m in EVAL_MODELS])
print(view.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\npaired accuracy deltas (boosted - baseline):")
for run, r in model_cmp.items():
    print(f"  {run:26s} " + "  ".join(f"{p}: {d['accuracy']['delta']:+.3f} (p={d['accuracy']['p_two_sided']:.3f})"
                                       for p, d in r["paired"].items() if p in ("xgb - logreg", "xgb - svm", "hgb - svm")))

                               run   model  accuracy           ci95  macro_f1    auc
                    loo_5class_169 XGBoost    0.3604 [0.336, 0.385]    0.3541 0.6895
                    loo_5class_169     HGB    0.3509 [0.327, 0.376]    0.3465 0.6810
                    loo_5class_169  LogReg    0.3373 [0.316, 0.359]    0.3320 0.6628
                    loo_5class_169 SVM-RBF    0.3404 [0.317, 0.364]    0.3367 0.6574
                  loo_4class_pa1ex XGBoost    0.4401 [0.413, 0.468]    0.4368 0.7148
                  loo_4class_pa1ex     HGB    0.4346 [0.406, 0.464]    0.4320 0.7085
                  loo_4class_pa1ex  LogReg    0.4197 [0.395, 0.445]    0.4145 0.6845
                  loo_4class_pa1ex SVM-RBF    0.4206 [0.393, 0.448]    0.4197 0.6857
               loo_5class_nofacial XGBoost    0.3271 [0.305, 0.350]    0.3200 0.6607
               loo_5class_nofacial     HGB    0.3202 [0.298, 0.343]    0.3154 0.6531
               loo_5class_nofacial  LogReg    0.3170 [0.295, 0.33

In [24]:
# ---------------- our features vs the published features ----------------
def compare_feature_sets(task):
    bo, labels = load_all_predictions(f"cmp_{task}_ours")
    bp, labels_p = load_all_predictions(f"cmp_{task}_plos")
    assert labels == labels_p and np.array_equal(bo["xgb"]["sample_id"], bp["xgb"]["sample_id"])
    assert np.array_equal(bo["xgb"]["true_class"], bp["xgb"]["true_class"])
    blocks = {**{f"ours:{m}": bo[m] for m in EVAL_MODELS}, **{f"plos:{m}": bp[m] for m in EVAL_MODELS}}
    blocks["xgb"] = bo["xgb"]                                            # reference block for score_block
    y, subject, preds, point, auc = score_block(blocks, labels)
    preds.pop("xgb"); point.pop("xgb"); auc.pop("xgb")
    draws = bootstrap_scores(subject, y, preds, len(labels))
    rows = {}
    for m in EVAL_MODELS:
        o, p = f"ours:{m}", f"plos:{m}"
        rows[m] = {"name": MODEL_NAMES[m], "ours": {**point[o], "auc": auc[o], "ci95": ci_of(draws, o)},
                   "published": {**point[p], "auc": auc[p], "ci95": ci_of(draws, p)},
                   "ours_minus_published": paired_delta(draws, point, o, p)}
    best = {fs: max(EVAL_MODELS, key=lambda m: point[f"{fs}:{m}"]["accuracy"]) for fs in ("ours", "plos")}
    best_delta = paired_delta(draws, point, f"ours:{best['ours']}", f"plos:{best['plos']}")
    return {"task": task, "label": COMPARISON_TASKS[task]["label"], "n_samples": int(len(y)),
            "n_subjects": int(len(np.unique(subject))), "n_classes": len(labels), "chance": CHANCE[len(labels)],
            "n_features": {"ours": len(COMPARISON_TASKS[task]["ours"]), "published": len(COMPARISON_TASKS[task]["plos"])},
            "models": rows, "best_model": best, "best_ours_minus_best_published": best_delta}


if HAVE_PLOS:
    fs_cmp = {task: compare_feature_sets(task) for task in COMPARISON_TASKS}
    md = ["# Our features vs the published Gruss et al. (2015) features\n",
          f"{plos_info['subjects_common']} subjects present in both sources, {plos_info['trials_matched']} trials "
          "retained by our quality filter; identical LOSO folds, within-subject z-score and models for both feature sets. "
          "Deltas are ours minus published, with 95% CIs from 2000 paired subject-level bootstrap resamples.\n",
          "| task | features (ours / published) | model | ours accuracy [95% CI] | published accuracy [95% CI] | delta | 95% CI | p | AUC ours / published |",
          "|---|---|---|---|---|---:|---|---:|---|"]
    rows = []
    for task, r in fs_cmp.items():
        for m, v in r["models"].items():
            o, p, d = v["ours"], v["published"], v["ours_minus_published"]["accuracy"]
            md.append(f"| {r['label']} | {r['n_features']['ours']} / {r['n_features']['published']} | {v['name']} | "
                      f"{o['accuracy']:.4f} [{o['ci95']['accuracy'][0]:.3f}, {o['ci95']['accuracy'][1]:.3f}] | "
                      f"{p['accuracy']:.4f} [{p['ci95']['accuracy'][0]:.3f}, {p['ci95']['accuracy'][1]:.3f}] | "
                      f"{d['delta']:+.4f} | [{d['ci95'][0]:+.3f}, {d['ci95'][1]:+.3f}] | {d['p_two_sided']:.3f} | "
                      f"{o['auc']:.3f} / {p['auc']:.3f} |")
            rows.append({"task": task, "model": v["name"], "ours": o["accuracy"], "published": p["accuracy"],
                         "delta": d["delta"], "ci95": "[{:+.3f}, {:+.3f}]".format(*d["ci95"]), "p": d["p_two_sided"]})
    md += ["\n## Protocol: random 75/25 trial split vs LOSO\n",
           "Gruss et al. (2015) reported 79.29% (BL1 vs PA1), 84.94% (BL1 vs PA3) and 90.94% (BL1 vs PA4) with an RBF-SVM, "
           f"a random 75/25 split and forward feature selection. The same split protocol ({SPLIT_REPEATS} repeats, "
           "no feature selection):\n",
           md_table(protocol_table)]
    md += ["\n## Audit of the published table\n",
           md_table(pd.DataFrame(plos_audit["duplication"]), "{:.3f}"), "",
           md_table(pd.DataFrame(plos_audit["facial_similarity_auc"]), "{:.3f}"), "",
           "## Link between published rows and released trials\n",
           md_table(pd.DataFrame(plos_info["link_check"]), "{:.3f}")]
    (REPORTS_DIR / "published_features_comparison.md").write_text("\n".join(md), encoding="utf-8")
    write_json(REPORTS_DIR / "published_features_comparison.json", {"link": plos_info, "audit": plos_audit, "tasks": fs_cmp})
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

                     task   model   ours  published   delta             ci95      p
                   5class XGBoost 0.3549     0.3906 -0.0357 [-0.051, -0.019] 0.0000
                   5class     HGB 0.3530     0.3848 -0.0317 [-0.046, -0.017] 0.0000
                   5class  LogReg 0.3364     0.3762 -0.0399 [-0.056, -0.024] 0.0000
                   5class SVM-RBF 0.3369     0.3823 -0.0455 [-0.059, -0.032] 0.0000
             4class_pa1ex XGBoost 0.4323     0.5333 -0.1010 [-0.120, -0.081] 0.0000
             4class_pa1ex     HGB 0.4379     0.5295 -0.0916 [-0.110, -0.073] 0.0000
             4class_pa1ex  LogReg 0.4156     0.5155 -0.0999 [-0.119, -0.080] 0.0000
             4class_pa1ex SVM-RBF 0.4167     0.5222 -0.1055 [-0.123, -0.086] 0.0000
               bl1_vs_pa1 XGBoost 0.5428     0.7039 -0.1611 [-0.191, -0.130] 0.0000
               bl1_vs_pa1     HGB 0.5646     0.7057 -0.1411 [-0.175, -0.109] 0.0000
               bl1_vs_pa1  LogReg 0.5526     0.7128 -0.1602 [-0.189, -0.130]

In [25]:
# ---------------- figures ----------------
MODEL_COLORS = {"xgb": "#2a78d6", "hgb": "#eb6834", "logreg": "#1baf7a", "svm": "#eda100"}   # CVD-checked palette
FS_COLORS = {"ours": "#6250d6", "plos": "#e87ba4"}


def bar_with_ci(ax, x, value, ci, color, width, label=None):
    ax.bar(x, value, width, color=color, label=label, edgecolor="white", linewidth=1.0)
    ax.errorbar(x, value, yerr=[[value - ci[0]], [ci[1] - value]], fmt="none", ecolor="#333333", elinewidth=0.9, capsize=2.5)
    ax.text(x, ci[1] + 0.012, f"{value:.2f}", ha="center", va="bottom", fontsize=7, color="#0b0b0b")


fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.4), sharey=True)
titles = {"loo_5class_169": "5-class, facial EMG without facial similarity", "loo_4class_pa1ex": "4-class (PA1 excluded), all features",
          "loo_5class_nofacial": "5-class, no facial EMG", "loo_4class_nofacial": "4-class (PA1 excluded), no facial EMG"}
for ax, run in zip(axes.ravel(), MAIN_RUNS):
    r = model_cmp[run]
    title = f"{titles[run]}\n{r['n_subjects']} subjects, {len(MAIN_RUNS[run][0])} features"
    for i, m in enumerate(EVAL_MODELS):
        v = r["models"][m]
        bar_with_ci(ax, i, v["accuracy"], v["ci95"]["accuracy"], MODEL_COLORS[m], 0.68)
    ax.axhline(CHANCE[r["n_classes"]], color="#52514e", lw=0.9, ls="--")
    ax.text(len(EVAL_MODELS) - 0.5, CHANCE[r["n_classes"]] + 0.01, "chance", ha="right", fontsize=7, color="#52514e")
    ax.set_xticks(range(len(EVAL_MODELS)), [MODEL_NAMES[m] for m in EVAL_MODELS])
    ax.set_title(title, fontsize=9.5)
    ax.grid(axis="y", alpha=0.25)
for ax in axes[:, 0]:
    ax.set_ylabel("LOSO accuracy (95% CI)")
axes[0, 0].set_ylim(0, 0.6)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig06_model_comparison.png", bbox_inches="tight")
plt.close(fig)

if HAVE_PLOS:
    base_tasks = ["5class", "4class"] + [f"bl1_vs_pa{k}" for k in (1, 2, 3, 4)]
    short = ["5-class", "4-class"] + [f"BL1 vs\nPA{k}" for k in (1, 2, 3, 4)]
    panels = [("", "facial EMG kept, facial similarity removed"), ("_nofacial", "no facial EMG"),
              ("_nofacial_nosim", "no facial EMG, no similarity features (strict like for like)")]
    fig, axes = plt.subplots(3, 1, figsize=(10.0, 10.0), sharex=True)
    for ax, (v, title) in zip(axes, panels):
        for i, base in enumerate(base_tasks):
            task = ("4class_pa1ex" if v == "" else f"4class{v}") if base == "4class" else f"{base}{v}"
            r = fs_cmp[task]["models"]["xgb"]
            for off, fs, key in ((-0.19, "ours", "ours"), (0.19, "plos", "published")):
                bar_with_ci(ax, i + off, r[key]["accuracy"], r[key]["ci95"]["accuracy"], FS_COLORS[fs], 0.36,
                            label=("our features" if fs == "ours" else "published features (Gruss et al. 2015)") if i == 0 else None)
            ch = fs_cmp[task]["chance"]
            ax.plot([i - 0.42, i + 0.42], [ch, ch], color="#52514e", lw=0.9, ls="--")
        ax.set_title(title, fontsize=9.5)
        ax.set_ylabel("XGBoost LOSO accuracy")
        ax.set_ylim(0, 0.97)
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(frameon=False, loc="upper left", fontsize=8, ncol=2)
    fig.suptitle(f"Same {plos_info['trials_matched']} trials, {plos_info['subjects_common']} subjects, folds and model; "
                 "dashed lines = chance", fontsize=10)
    axes[-1].set_xticks(range(len(base_tasks)), short)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig07_published_features.png", bbox_inches="tight")
    plt.close(fig)

for p in sorted(FIG_DIR.glob("fig0[67]*.png")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KiB)")

  fig06_model_comparison.png  (90 KiB)
  fig07_published_features.png  (115 KiB)


## Step 11 — summary for the paper

Collects every number the paper needs from the files written above into `outputs/reports/summary.md`
(data, features, models, integrity findings, main results, baselines, published-feature comparison, protocol effect,
feature importance, figures, limitations, software versions) and one row per run and model into `summary.csv`.
The cell reads files only, so it can be re-run on its own after the configuration cell.

In [26]:
import platform

import joblib
import matplotlib
import scipy
import sklearn
import xgboost


def rj(path):
    return json.loads(path.read_text(encoding="utf-8"))


def ci_txt(c):
    return f"[{c[0]:.3f}, {c[1]:.3f}]"


def p_txt(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def delta_txt(d):
    return f"{d['delta']:+.3f} (95% CI {d['ci95'][0]:+.3f} to {d['ci95'][1]:+.3f}, {p_txt(d['p_two_sided'])})"


def verdict(d):
    lo, hi = d["ci95"]
    return "higher" if lo > 0 else ("lower" if hi < 0 else "not different")


def table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "|".join("---" for _ in header) + "|"]
    out += ["| " + " | ".join(str(v) for v in r) + " |" for r in rows]
    return out


def task_short(task):
    base = task.replace("_nofacial_nosim", "").replace("_nofacial", "").replace("_pa1ex", "")
    return base.replace("5class", "5-class").replace("4class", "4-class").replace("bl1_vs_pa", "BL1 vs PA")


def channel_of(feature):
    f = feature.replace("plos_", "")
    return "EDA" if f.startswith("gsr") else ("ECG" if f.startswith("ecg") else "EMG " + f.split("_")[1])


NAMES = {"xgb": "XGBoost", "hgb": "HGB", "logreg": "LogReg", "svm": "SVM-RBF"}
NAMES.update({f"logreg_C{c:g}": f"LogReg C={c:g}" for c in (0.01, 0.1)})
NAMES.update({f"svm_C{c:g}": f"SVM-RBF C={c:g}" for c in (0.1, 10)})
MAIN = ["loo_5class_169", "loo_4class_pa1ex", "loo_5class_nofacial", "loo_4class_nofacial"]
RUN_LABELS = {
    "full_template_5class": "5-class, full BL1 template (contaminated)",
    "loo_5class": "5-class, leave-one-out template, 175 features (facial copy leaks)",
    "loo_5class_169": "5-class, 169 features (no facial similarity)",
    "loo_4class_pa1ex": "4-class BL1/PA2/PA3/PA4, 175 features",
    "loo_5class_nofacial": "5-class, no facial EMG",
    "loo_4class_nofacial": "4-class BL1/PA2/PA3/PA4, no facial EMG",
}
CHANCE_OF = {5: 0.20, 4: 0.25, 2: 0.50}
N_BOOT = globals().get("N_BOOT", 2000)
FACIAL_SIM_COLS = [f"emg_{ch}_sim_{k}" for ch in ("corrugator", "zygomaticus") for k in ("corr", "coh_mean", "mi")]

cache = rj(CACHE_DIR / "cache_manifest.json")
fraw = rj(FEATURES_DIR / "features_raw_manifest.json")
fmod = rj(FEATURES_DIR / "features_model_manifest.json")
index = pd.read_parquet(FEATURES_DIR / "index_biosignals_partB.parquet")
manifests = {p.name[len("loso_"):-len("_manifest.json")]: rj(p) for p in sorted(MODELS_DIR.glob("loso_*_manifest.json"))
             if "__baselines" not in p.name}
evals = {run: rj(REPORTS_DIR / f"eval_{run}.json") for run in MAIN if (REPORTS_DIR / f"eval_{run}.json").exists()}
tcorr = rj(REPORTS_DIR / "template_correction.json")
dup = rj(REPORTS_DIR / "pa1_duplication.json")
mcmp = rj(REPORTS_DIR / "model_comparison.json")
have_plos = (REPORTS_DIR / "published_features_comparison.json").exists() and PLOS_FEATURES_FILE is not None
if have_plos:
    pcmp = rj(REPORTS_DIR / "published_features_comparison.json")
    paudit, plink, ptasks = pcmp["audit"], pcmp["link"], pcmp["tasks"]
    proto_json = rj(REPORTS_DIR / "protocol_random_split.json")
    proto = pd.DataFrame(proto_json["rows"])
    PUBLISHED_ACCURACY, SPLIT_REPEATS = proto_json["published_accuracy_gruss_2015"], proto_json["repeats"]
kept = fmod["feature_columns"]
n_nofacial = sum(not c.startswith(("emg_corrugator_", "emg_zygomaticus_")) for c in kept)
per_subject = index[index["keep"]].groupby("subject_name").size()

L = ["# BioVid Part B — results summary for the paper\n",
     "Generated by Step 11 of `biovid_partB_pipeline.ipynb` from the files in `outputs/`; every number below is read "
     "from those files. Accuracy is the proportion of trials classified correctly. Chance is 0.20 (5 classes), 0.25 "
     f"(4 classes) and 0.50 (2 balanced classes). 95% CIs come from {N_BOOT} subject-level bootstrap resamples; "
     "p values are two-sided paired bootstrap tests on the same resamples.\n"]

# ================= 1. headline numbers =================
L.append("## 1. Headline numbers\n")
for run in MAIN:
    if run not in evals:
        continue
    e = evals[run]
    parts = []
    for m in MODELS:
        c, b = e["models"][m]["core"], e["models"][m]["bootstrap"]
        parts.append(f"{MODEL_NAMES[m]} accuracy {c['accuracy']:.3f} {ci_txt(b['accuracy']['ci95'])}, macro-F1 "
                     f"{c['macro_f1']:.3f}, quadratic kappa {c['kappa_quadratic']:.3f}")
    L.append(f"- **{RUN_LABELS[run]}** ({e['n_subjects']} subjects, {e['n_samples']} trials, chance "
             f"{CHANCE_OF[len(e['class_counts'])]:.2f}): " + "; ".join(parts) + ".")
e = evals["loo_5class_169"]
L.append(f"- **Baseline vs PA4** (from the 5-class, 169-feature probabilities): AUC "
         f"{e['models']['xgb']['binary']['BL1_vs_PA4']['auc']:.3f} (XGBoost); **baseline vs PA1**: AUC "
         f"{e['models']['xgb']['binary']['BL1_vs_PA1']['auc']:.3f}; **baseline vs any pain**: AUC "
         f"{e['models']['xgb']['binary']['BL1_vs_any_pain']['auc']:.3f}.")
for b in ("logreg", "svm"):
    wins = [run for run in MAIN if verdict(mcmp[run]["paired"][f"xgb - {b}"]["accuracy"]) == "higher"]
    L.append(f"- **XGBoost vs {NAMES[b]} (C = 1)**: accuracy significantly higher in {len(wins)} of {len(MAIN)} main runs; "
             "deltas " + "; ".join(f"{RUN_LABELS[run]} {delta_txt(mcmp[run]['paired'][f'xgb - {b}']['accuracy'])}"
                                    for run in MAIN) + ".")
fc = "BL1" if not have_plos else paudit["copied_class"]
L.append(f"- **Release defect**: in {dup['per_channel']['emg_corrugator']['pooled_pa1_matched']} of "
         f"{dup['per_channel']['emg_corrugator']['pooled_pa1_total']} PA1 trials the corrugator signal is byte-identical to a "
         f"BL1 trial of the same subject (zygomaticus: {dup['per_channel']['emg_zygomaticus']['pooled_pa1_matched']}). "
         + (f"Against the features Gruss et al. computed from the original recordings, the copied class is **{fc}**: the "
            "released BL1 facial EMG are PA1 recordings." if have_plos and fc == "BL1" else
            f"Direction of the copy: {fc}."))
r = {row["model"]: row for row in tcorr["full_template_vs_loo"]}
L.append(f"- **Self-inclusion in the BL1 template** inflates 5-class accuracy from {r['XGBoost']['acc_loo']:.3f} to "
         f"{r['XGBoost']['acc_full_template']:.3f} (XGBoost); removing the six facial similarity features lowers it further to "
         f"{r['XGBoost']['acc_loo_169']:.3f}.")
if have_plos:
    for suffix, text in (("", "facial EMG kept, facial similarity removed"), ("_nofacial", "no facial EMG"),
                         ("_nofacial_nosim", "no facial EMG and no similarity features, strict like for like")):
        group = [t for t in ptasks if (t.endswith(suffix) if suffix else not t.endswith(("_nofacial", "_nofacial_nosim")))]
        cnt = {k: sum(verdict(ptasks[t]["models"]["xgb"]["ours_minus_published"]["accuracy"]) == k for t in group)
               for k in ("higher", "lower", "not different")}
        L.append(f"- **Our vs the published features, {text}** ({plink['subjects_common']} subjects, {plink['trials_matched']} "
                 f"trials, XGBoost on both): ours significantly higher in {cnt['higher']}, lower in {cnt['lower']}, not "
                 f"different in {cnt['not different']} of {len(group)} tasks (ours vs published: " + "; ".join(
                     f"{task_short(t)} {ptasks[t]['models']['xgb']['ours']['accuracy']:.3f} vs "
                     f"{ptasks[t]['models']['xgb']['published']['accuracy']:.3f}" for t in group) + ").")
    a, b = ptasks["bl1_vs_pa1_nofacial"]["models"]["xgb"], ptasks["bl1_vs_pa1_nofacial_nosim"]["models"]["xgb"]
    L.append(f"- **The published similarity features carry the published advantage**: without facial EMG, BL1 vs PA1 accuracy "
             f"(XGBoost) of the published features falls from {a['published']['accuracy']:.3f} to {b['published']['accuracy']:.3f} "
             f"when their similarity features are removed, while ours moves from {a['ours']['accuracy']:.3f} to "
             f"{b['ours']['accuracy']:.3f}. Their similarity features separate BL1 from pain on channels where a "
             "leave-one-out template shows no effect (Section 5.3).")
    pr = proto[(proto["task"] == "bl1_vs_pa4") & (proto["features"] == "plos") & (proto["model"] == "svm")].iloc[0]
    max_gap = float(proto["split_minus_loso"].abs().max())
    L.append(f"- **Protocol effect**: with within-subject z-scoring, a random 75/25 trial split and LOSO give nearly the same "
             f"accuracy (largest difference {max_gap:.3f} over all tasks, feature sets and models). On the published features "
             f"an RBF-SVM reaches {pr['random_split_acc']:.3f} (split) and {pr['loso_acc']:.3f} (LOSO) for BL1 vs PA4, against "
             f"the published {PUBLISHED_ACCURACY['bl1_vs_pa4']:.4f}. Subject overlap alone therefore does not explain the "
             "published accuracies; their forward feature selection and SVM grid search were not reproduced here.")
L.append("")

# ================= 2. data =================
L.append("## 2. Data\n")
excl = cache["excluded_by_reason"]
L += table(["item", "value"], [
    ["source", "BioVid Heat Pain Database, Part B, filtered biosignals"],
    ["channels", "EDA (GSR), ECG, EMG trapezius, EMG corrugator, EMG zygomaticus"],
    ["sampling rate", f"{cache['fs_hz_median']:.2f} Hz (from microsecond time stamps)"],
    ["window", f"{SEG_LEN} samples = {SEG_LEN / cache['fs_hz_median']:.2f} s per trial"],
    ["subjects", f"{index['subject_name'].nunique()}"],
    ["trials in the release", f"{cache['n_index_rows']} (100 per subject: 20 per class)"],
    ["excluded trials", f"{cache['n_excluded']} ({100 * cache['n_excluded'] / cache['n_index_rows']:.1f}%): NaN/corrupt "
                        f"{excl['corrupt_or_nan']}, flat channel " +
                        ", ".join(f"{k[5:]} {v}" for k, v in excl.items() if k.startswith("flat_") and v)],
    ["retained trials", f"{cache['n_cached']}: " + ", ".join(f"{k} {v}" for k, v in cache["per_class_cached"].items())],
    ["trials per subject", f"min {per_subject.min()}, median {per_subject.median():.0f}, max {per_subject.max()}"],
    ["exclusion rule", f"any NaN, or any channel with std <= {FLAT_EPS:g}"],
])
L.append("\n84 of the 86 Part B subjects also appear in Part A, so Part B is not an independent cohort for Part A models.\n")

# ================= 3. features =================
L.append("## 3. Features\n")
mod_counts = fraw["features_per_modality"]
kept_by = pd.Series([channel_of(c) for c in kept]).value_counts()
L += table(["modality", "extracted", "kept after hygiene"], [
    ["EDA (GSR)", mod_counts["eda"], int(kept_by.get("EDA", 0))],
    ["ECG", mod_counts["ecg"], int(kept_by.get("ECG", 0))],
    ["EMG (3 channels)", mod_counts["emg"], int(sum(v for k, v in kept_by.items() if k.startswith("EMG")))],
    ["total", fraw["n_features"], fmod["n_features_out"]],
])
L += ["",
      "- **EDA**: statistics of the raw signal; tonic component by a 2nd-order Butterworth low-pass at 0.05 Hz (zero phase), "
      "phasic = raw - tonic; Hjorth parameters; SCR-like peaks by prominence 0.10 on the z-scored signal with 0.5 s minimum "
      "distance; Welch PSD of the phasic signal (nperseg 2048) with band powers 0.05-0.5, 0.5-1.5 and 1.5-5 Hz and spectral shape.",
      "- **ECG**: amplitude statistics, Hjorth, Welch PSD (nperseg 1024) with bands 0.5-5, 5-15, 15-40 Hz; R peaks from a "
      "5-20 Hz band-pass envelope (80 ms smoothing, 90th-percentile threshold raised while the rate exceeds 180 bpm, 0.3 s "
      "refractory); time-domain HRV only (5.5 s windows do not support LF/HF).",
      "- **EMG** (each channel, after 50 and 100 Hz notch filters, Q = 30): amplitude, RMS, MAV, waveform length, zero "
      "crossings, slope sign changes, Hjorth, Welch PSD (nperseg 1024) with bands 10-30, 30-60, 60-100, 100-150 Hz and spectral "
      "shape, burst statistics over 20 RMS windows, STFT spectral flux.",
      "- **Baseline similarity** (every channel): Pearson correlation, magnitude-squared coherence (mean, median, PSD-weighted) "
      "and binned mutual information between the trial and the subject's mean BL1 signal. The template excludes the trial "
      "itself (leave-one-out) in every reported run except `full_template_5class`.",
      f"- **Hygiene**: {len(fmod['constant_features'])} constant features dropped, then {fmod['n_duplicate_dropped']} dropped by a "
      f"greedy pass at |r| > {fmod['dedupe']['threshold']} on within-subject z-scores, leaving {fmod['n_features_out']}.",
      f"- **Feature sets used**: 175 (all kept), 169 (without the six facial similarity features "
      f"{', '.join('`' + c + '`' for c in FACIAL_SIM_COLS)}), and {n_nofacial} without any corrugator or zygomaticus feature.",
      ""]

# ================= 4. models and validation =================
L.append("## 4. Models and validation\n")
L += [f"- **Validation**: leave-one-subject-out ({index['subject_name'].nunique()} folds); every model sees identical folds.",
      "- **Normalisation**: each feature is z-scored within subject using all of that subject's trials (labels unused); "
      "features constant within a subject are set to 0.",
      "- **XGBoost**: " + ", ".join(f"{k}={v}" for k, v in XGB_PARAMS.items() if k not in ("n_jobs",)) +
      " (binary tasks: objective binary:logistic). Parameters come from Part A and were not tuned on Part B.",
      "- **HistGradientBoosting (HGB)**: " + ", ".join(f"{k}={v}" for k, v in HGB_PARAMS.items()) + " (capacity-matched to XGBoost).",
      "- **Logistic regression**: multinomial, L2, lbfgs, C = 1 (grid 0.01, 0.1, 1 on the main runs).",
      "- **SVM**: RBF kernel, gamma = 'scale', one-vs-one, C = 1 (grid 0.1, 1, 10 on the main runs); AUC uses a softmax of "
      "the decision values.",
      "- **No class weighting, no feature selection, single seed "
      f"({SEED})**. Classes are balanced by design (20 trials per class and subject before exclusions).",
      f"- **Statistics**: accuracy, macro-F1, quadratic and unweighted Cohen's kappa, MAE in levels, Spearman rho, within-one-level "
      f"rate, one-vs-rest AUC; {N_BOOT} subject-level bootstrap resamples for CIs; paired bootstrap for model and feature-set "
      "differences.",
      ""]

# ================= 5. integrity =================
L.append("## 5. Data-integrity findings\n")
L.append("### 5.1 Self-inclusion in the baseline template\n")
L += table(["model", "full template", "leave-one-out", "leave-one-out, 169 features"],
           [[row["model"], f"{row['acc_full_template']:.4f}", f"{row['acc_loo']:.4f}", f"{row['acc_loo_169']:.4f}"]
            for row in tcorr["full_template_vs_loo"]])
L += ["", "BL1-vs-rest AUC of single similarity features (within-subject z-scored):", ""]
L += table(["feature", "full template", "leave-one-out"],
           [[row["feature"], f"{row['auc_bl1_full_template']:.3f}", f"{row['auc_bl1_loo_template']:.3f}"]
            for row in tcorr["similarity_feature_auc"]])
L += ["", "With the full template every BL1 trial is part of its own reference, so its similarity is inflated and BL1 "
      "becomes trivially separable. The leave-one-out template removes this.", ""]

L.append("### 5.2 Facial-EMG duplication between BL1 and PA1 in the release\n")
L += table(["channel", "subjects with BL1 set = PA1 set", "PA1 trials identical to a BL1 trial", "minimum per-subject match"],
           [[ch, f"{v['subjects_set_equal']} / {dup['n_subjects']}", f"{v['pooled_pa1_matched']} / {v['pooled_pa1_total']} "
             f"({100 * v['pooled_pa1_matched'] / v['pooled_pa1_total']:.1f}%)", f"{v['min_pa1_match_fraction']:.3f}"]
            for ch, v in dup["per_channel"].items()])
tw = dup["twin_pair_predictions_loo_5class"]
L += ["", f"Source CSVs confirm it (`{dup['source_file_check']['files'][0]}` vs `{dup['source_file_check']['files'][1]}`: "
      + ", ".join(f"{ch} {'identical' if eq else 'different'}" for ch, eq in dup['source_file_check']['byte_identical'].items())
      + f"). In the leave-one-out 5-class run with all 175 features, the {dup['n_twin_pairs']} twin pairs are both classified "
      f"correctly in {tw['xgb']['both_correct']:.1%} (XGBoost) of cases and the PA1 twin receives the higher PA1 probability in "
      f"{tw['xgb']['pa1_prob_higher_for_pa1_twin']:.1%}: the facial similarity features act as a PA1 detector.", ""]
if have_plos:
    L += ["**Which class holds the copy.** Within-subject correlation between our EMG amplitude (RMS) and the published "
          "amplitude (A-RMS), which Gruss et al. computed from the original recordings:", ""]
    L += table(["channel"] + [CLASS_NAMES[k] for k in LABELS],
               [[row["channel"]] + [f"{row[CLASS_NAMES[k]]:.3f}" for k in LABELS] for row in paudit["per_class_agreement"]])
    dd = {row["channel"]: row for row in paudit["duplication"]}
    L += ["", "Facial agreement is high in PA1-PA4 and low only in BL1, while trapezius agrees in every class.", ""]
    L += table(["channel", "published PA1 rows identical to a BL1 row", "twin r (ours)", "twin r (published BL1 vs PA1 rows)"],
               [[row["channel"], f"{row['published_pa1_rows_identical_to_a_bl1_row']} / {row['published_pa1_rows']}",
                 f"{row['twin_r_ours_rms']:.3f}", f"{row['twin_r_published_rms']:.3f}"] for row in paudit["duplication"]])
    L += ["", f"For the released twin pairs our amplitudes are identical by construction (r = 1), while the published BL1 and "
          f"PA1 rows of the same pairs correlate only {dd['corrugator']['twin_r_published_rms']:.3f} (corrugator) and "
          f"{dd['zygomaticus']['twin_r_published_rms']:.3f} (zygomaticus), as expected for two different recordings. "
          f"Conclusion: **the copied class is {paudit['copied_class']}**. The released BL1 corrugator and zygomaticus signals "
          "are PA1 recordings, while the PA1-PA4 facial signals are genuine. Consequences: (i) facial similarity features leak "
          "(above); (ii) in every run that keeps facial EMG, the BL1 facial features describe PA1 trials, so facial information "
          "about baseline vs pain is lost. The no-facial runs avoid both problems.", ""]
    sep = paudit["max_separation"]
    L += [f"Largest single-feature BL1-vs-PA1 separation (|2 AUC - 1|) of the facial similarity features: ours "
          f"{sep.get('ours (leave-one-out template)', float('nan')):.3f}, published {sep.get('published', float('nan')):.3f}.", ""]
    L.append("### 5.3 Self-inclusion in the published similarity features\n")
    ms = paudit["similarity_self_inclusion_max_separation"]
    sets = ["ours, full template", "ours, leave-one-out", "published"]
    L += ["Largest BL1-vs-PA2..PA4 separation |2 AUC - 1| of a single similarity feature per channel (PA1 left out because "
          "of the facial copy; within-subject z-scored; matched trials):", ""]
    fmt = lambda v: "n/a" if v is None or pd.isna(v) else f"{v:.3f}"
    L += table(["channel"] + sets, [[ch] + [fmt(ms[ch].get(s)) for s in sets] for ch in ms])
    L += ["", "Where our leave-one-out features show no baseline effect (ECG, trapezius), our full-template features and "
          "the published features still separate BL1 from pain. This is the signature of a baseline template that "
          "contains the trial itself, which the paper's description (similarity to the person's mean baseline signal) implies. "
          "The `nofacial_nosim` comparison therefore removes similarity features from both sets.", ""]

# ================= 6. main results =================
L.append("## 6. Main results, all subjects (LOSO)\n")
rows = []
for run in MAIN:
    if run not in evals:
        continue
    e = evals[run]
    for m in MODELS:
        c, b = e["models"][m]["core"], e["models"][m]["bootstrap"]
        rows.append([RUN_LABELS[run], len(e["class_counts"]), manifests[run]["n_features"], MODEL_NAMES[m],
                     f"{c['accuracy']:.4f} {ci_txt(b['accuracy']['ci95'])}", f"{c['macro_f1']:.4f} {ci_txt(b['macro_f1']['ci95'])}",
                     f"{c['kappa_quadratic']:.4f} {ci_txt(b['kappa_quadratic']['ci95'])}", f"{c['mae']:.3f}",
                     f"{c['spearman']:.3f}", f"{c['within_one_level']:.3f}"])
L += table(["run", "classes", "features", "model", "accuracy [95% CI]", "macro-F1 [95% CI]", "quadratic kappa [95% CI]",
            "MAE", "Spearman", "within one level"], rows)
L += ["", "Contaminated runs, for reference only: " + "; ".join(
    f"{RUN_LABELS[run]}: XGBoost {manifests[run]['metrics']['xgb']['accuracy']:.4f}, HGB {manifests[run]['metrics']['hgb']['accuracy']:.4f}"
    for run in ("full_template_5class", "loo_5class") if run in manifests) + ".", ""]

L.append("**Per-class results (XGBoost)**\n")
rows = []
for run in MAIN:
    if run in evals:
        for pc in evals[run]["models"]["xgb"]["per_class"]:
            rows.append([RUN_LABELS[run], pc["class"], pc["support"], f"{pc['precision']:.3f}", f"{pc['recall']:.3f}",
                         f"{pc['f1']:.3f}", f"{pc['auc_ovr']:.3f}"])
L += table(["run", "class", "n", "precision", "recall", "F1", "AUC (one vs rest)"], rows)

L.append("\n**Binary tasks derived from the multiclass probabilities** (AUC, balanced accuracy at 0.5)\n")
rows = []
for run in MAIN:
    if run in evals:
        for m in MODELS:
            for task, v in evals[run]["models"][m]["binary"].items():
                rows.append([RUN_LABELS[run], MODEL_NAMES[m], task.replace("_", " "), v["n_samples"], f"{v['auc']:.3f}",
                             f"{v['balanced_accuracy']:.3f}", f"{v['sensitivity']:.3f}", f"{v['specificity']:.3f}"])
L += table(["run", "model", "task", "n", "AUC", "balanced accuracy", "sensitivity", "specificity"], rows)

L.append("\n**XGBoost minus HGB (paired)**\n")
L += table(["run", "accuracy", "macro-F1", "quadratic kappa"],
           [[RUN_LABELS[run]] + [delta_txt({"delta": evals[run]["paired_xgb_minus_hgb"][k]["mean_delta"],
                                            "ci95": evals[run]["paired_xgb_minus_hgb"][k]["ci95"],
                                            "p_two_sided": evals[run]["paired_xgb_minus_hgb"][k]["p_two_sided"]})
                                 for k in ("accuracy", "macro_f1", "kappa_quadratic")] for run in MAIN if run in evals])
L.append("")

# ================= 7. baselines =================
L.append("## 7. Boosted models vs linear baselines\n")
L.append("Same folds, features and normalisation for all four models (main runs, all subjects).\n")
rows = []
for run in MAIN:
    r = mcmp[run]
    for m in ["xgb", "hgb", "logreg", "svm"]:
        v = r["models"][m]
        rows.append([RUN_LABELS[run], NAMES[m], f"{v['accuracy']:.4f} {ci_txt(v['ci95']['accuracy'])}",
                     f"{v['macro_f1']:.4f}", f"{v['kappa_quadratic']:.4f}", f"{v['auc']:.3f}"])
L += table(["run", "model", "accuracy [95% CI]", "macro-F1", "quadratic kappa", "AUC (macro one vs rest)"], rows)
L.append("\n**Paired differences (boosted minus baseline, C = 1)**\n")
rows = []
for run in MAIN:
    for pair in ("xgb - logreg", "xgb - svm", "hgb - logreg", "hgb - svm"):
        d = mcmp[run]["paired"][pair]
        a, b = pair.split(" - ")
        rows.append([RUN_LABELS[run], f"{NAMES[a]} - {NAMES[b]}", delta_txt(d["accuracy"]), delta_txt(d["macro_f1"]),
                     delta_txt(d["kappa_quadratic"])])
L += table(["run", "comparison", "accuracy", "macro-F1", "quadratic kappa"], rows)
L.append("\n**Sensitivity to the baseline's C** (accuracy; the best C is chosen on the test folds, which favours the baseline)\n")
rows = []
for run in MAIN:
    r = mcmp[run]
    keys = [k for k in NAMES if k in r["models"] and k not in ("xgb", "hgb")]
    best = {kind: r["best_grid_baseline"][kind] for kind in ("logreg", "svm")}
    worst_gap = min(r["models"]["xgb"]["accuracy"] - r["models"][best[k]]["accuracy"] for k in best)
    rows.append([RUN_LABELS[run]] + [f"{r['models'][k]['accuracy']:.4f}" for k in keys] +
                [f"{r['models']['xgb']['accuracy']:.4f}", f"{worst_gap:+.4f}"])
L += table(["run"] + [NAMES[k] for k in NAMES if k not in ("xgb", "hgb")] + ["XGBoost", "XGBoost - best baseline"], rows)
best_pairs = []
for run in MAIN:
    for pair, d in mcmp[run]["paired"].items():
        if "_C" in pair and pair.startswith("xgb"):
            best_pairs.append(f"{RUN_LABELS[run]}, {NAMES[pair.split(' - ')[1]]}: {delta_txt(d['accuracy'])}")
if best_pairs:
    L += ["", "XGBoost vs the best-C baseline: " + "; ".join(best_pairs) + "."]
L.append("")

# ================= 8. published features =================
if have_plos:
    L.append("## 8. Our features vs the published feature set of Gruss et al. (2015)\n")
    L += [f"- Published table: {plink['n_features']} features, {plink['subjects_published']} subjects x 100 trials. "
          f"Common subjects: {plink['subjects_common']} (only published: {', '.join(plink['subjects_only_published']) or 'none'}; "
          f"only in our release: {', '.join(plink['subjects_only_ours']) or 'none'}).",
          f"- Rows were linked to released trials by chronological rank within subject and class; the pain-level sequence is "
          f"identical in both sources for all {plink['subjects_common']} subjects. {plink['trials_linked']} trials linked, "
          f"{plink['trials_matched']} kept after our quality filter ({plink['trials_dropped_by_our_quality_filter']} dropped).",
          "- Missing published values were filled with the subject's median: " +
          ", ".join(f"{k} {v}" for k, v in plink["nan_cells_imputed"].items()) + ".",
          "- Both feature sets then go through the identical within-subject z-score, LOSO folds and models.", ""]
    L += ["Link check (within-subject Pearson r between quantities both pipelines measure; a one-trial shift is the control):", ""]
    L += table(["ours", "published", "r BL1", "r pain", "r shifted by one trial"],
               [[row["ours"], row["published"], f"{row['r_bl1']:.3f}", f"{row['r_pain']:.3f}", f"{row['r_shifted_by_one_trial']:.3f}"]
                for row in plink["link_check"]])
    L += ["", "**LOSO accuracy, ours vs published** (delta = ours - published, paired subject bootstrap)", ""]
    rows = []
    for task, t in ptasks.items():
        for m in ("xgb", "hgb", "logreg", "svm"):
            v = t["models"][m]
            rows.append([t["label"], f"{t['n_features']['ours']} / {t['n_features']['published']}", NAMES[m],
                         f"{v['ours']['accuracy']:.4f} {ci_txt(v['ours']['ci95']['accuracy'])}",
                         f"{v['published']['accuracy']:.4f} {ci_txt(v['published']['ci95']['accuracy'])}",
                         delta_txt(v["ours_minus_published"]["accuracy"]), f"{v['ours']['auc']:.3f} / {v['published']['auc']:.3f}"])
    L += table(["task", "features (ours / published)", "model", "ours", "published", "delta", "AUC ours / published"], rows)
    L += ["", "**Best model of each feature set**", ""]
    L += table(["task", "best (ours)", "best (published)", "delta accuracy", "verdict"],
               [[t["label"], NAMES[t["best_model"]["ours"]], NAMES[t["best_model"]["plos"]],
                 delta_txt(t["best_ours_minus_best_published"]["accuracy"]),
                 "ours " + verdict(t["best_ours_minus_best_published"]["accuracy"])] for t in ptasks.values()])
    L += ["", "**Validation protocol.** Gruss et al. (2015) used an RBF-SVM with grid search and forward feature selection on a "
          "random 75/25 trial split, so the same subjects appear in training and test data, and reported "
          + ", ".join(f"{k.replace('bl1_vs_pa', 'BL1 vs PA')} {v:.4f}" for k, v in PUBLISHED_ACCURACY.items())
          + f". The same split protocol without feature selection ({SPLIT_REPEATS} repeats) compared with LOSO:", ""]
    L += table(["task", "features", "model", "random split (mean +- sd)", "LOSO", "split - LOSO", "published (split + selection)"],
               [[row["task"], "published" if row["features"] == "plos" else row["features"], NAMES[row["model"]], f"{row['random_split_acc']:.4f} +- {row['random_split_sd']:.4f}",
                 f"{row['loso_acc']:.4f}", f"{row['split_minus_loso']:+.4f}",
                 "" if pd.isna(row["published_acc"]) else f"{row['published_acc']:.4f}"] for _, row in proto.iterrows()])
    L.append("")

# ================= 9. feature importance =================
L.append("## 9. Feature importance (mean XGBoost gain share over LOSO folds)\n")
for run in ("loo_5class_169", "loo_5class_nofacial"):
    g = pd.read_parquet(MODELS_DIR / f"xgb_gain_{run}.parquet")
    share = g.assign(channel=g["feature"].map(channel_of)).groupby("channel")["mean_gain"].sum().sort_values(ascending=False)
    L += [f"**{RUN_LABELS[run]}**: gain share by channel: " + ", ".join(f"{k} {v:.3f}" for k, v in share.items()), ""]
    L += table(["rank", "feature", "gain share"], [[int(r_["rank"]), r_["feature"], f"{r_['mean_gain']:.4f}"]
                                                  for _, r_ in g.head(15).iterrows()])
    L.append("")
if have_plos:
    g = pd.read_parquet(MODELS_DIR / "xgb_gain_cmp_5class_nofacial_plos.parquet")
    share = g.assign(channel=g["feature"].map(channel_of)).groupby("channel")["mean_gain"].sum().sort_values(ascending=False)
    L += ["**Published features, 5-class, no facial EMG**: gain share by channel: " +
          ", ".join(f"{k} {v:.3f}" for k, v in share.items()) + "; top 10: " +
          ", ".join(f"{f} ({v:.3f})" for f, v in g.head(10)[["feature", "mean_gain"]].to_numpy()) + ".", ""]

# ================= 10. figures =================
L.append("## 10. Figures (`outputs/figures/`)\n")
L += ["- `fig01_confusion_5class_169.png`: row-normalised confusion matrices, 5-class, 169 features, XGBoost and HGB.",
      "- `fig02_confusion_4class_pa1ex.png`: the same for the 4-class task (PA1 excluded).",
      "- `fig03_per_subject_accuracy.png`: per-subject LOSO accuracy (5-class, 169 features) and the paired XGBoost/HGB scatter.",
      "- `fig04_importance_top20.png`: top-20 XGBoost gain with (175) and without (169) the facial similarity features.",
      "- `fig05_correction_history.png`: 5-class scores after each correction (full template, leave-one-out, 169 features).",
      "- `fig06_model_comparison.png`: accuracy with 95% CIs of XGBoost, HGB, logistic regression and RBF-SVM on the four main runs."]
if have_plos:
    L.append("- `fig07_published_features.png`: XGBoost LOSO accuracy with our vs the published features per task, in the "
             "three variants (facial EMG kept; no facial EMG; no facial EMG and no similarity features).")
L.append("")

# ================= 11. limitations =================
L.append("## 11. Limitations and points for the discussion\n")
L += ["- Within-subject z-scoring uses all of the test subject's trials (without labels); this is standard for BioVid but is "
      "mildly optimistic compared with training-fold-only scaling, and it assumes a calibration recording per subject.",
      "- Hyperparameters were fixed in advance (XGBoost from Part A; HGB capacity-matched; baselines at library defaults with a "
      "small C grid). No nested tuning was run.",
      "- Results use one seed; the subject bootstrap captures between-subject variance, not seed variance.",
      "- 5.5 s windows only allow time-domain HRV; ECG contributes little at this length.",
      f"- The released facial EMG of BL1 is a copy of PA1 (Section 5.2). Runs that keep facial EMG therefore describe PA1 faces "
      "in the BL1 class; the no-facial runs are the cleanest estimate but lose the most informative channels reported in the literature.",
      "- Part B shares 84 of 86 subjects with Part A.",
      "- The comparison with Gruss et al. uses their features but not their feature selection or SVM grid search; their "
      "accuracies were obtained with a random trial split and are not comparable with LOSO numbers.",
      "- The published similarity features behave as if each BL1 trial were part of its own baseline template (Section 5.3); "
      "this cannot be confirmed without their code, so the strict comparison simply excludes all similarity features.",
      "- The link between published rows and released trials relies on chronological order; it is supported by within-subject "
      "correlations of 0.8-0.97 on intact channels against 0.16-0.36 for a one-trial shift."]
L.append("")

# ================= 12. reproducibility =================
L.append("## 12. Reproducibility\n")
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "joblib": joblib.__version__,
            "matplotlib": matplotlib.__version__}
L += [f"- Software: " + ", ".join(f"{k} {v}" for k, v in versions.items()) + f"; {platform.system()} build "
      f"{platform.version()}, {os.cpu_count()} logical CPUs.",
      f"- Seed {SEED}; folds defined by subject; XGBoost hist method (last-digit differences across CPUs are possible).",
      "- Run: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 biovid_partB_pipeline.ipynb`.",
      "- LOSO runtime per run (seconds): " + ", ".join(f"{run} {m['runtime_seconds']['total']:.0f}" for run, m in manifests.items()) + "."]
(REPORTS_DIR / "summary.md").write_text("\n".join(L) + "\n", encoding="utf-8")

# ================= summary.csv: one row per run and model =================
rows = []
for run, man in manifests.items():
    base = MODELS_DIR / f"loso_{run}__baselines_manifest.json"
    metrics = {**man["metrics"], **(rj(base)["metrics"] if base.exists() else {})}
    for m, c in metrics.items():
        ci = (mcmp.get(run, {}).get("models", {}).get(m, {}).get("ci95", {}).get("accuracy")
              or (evals[run]["models"][m]["bootstrap"]["accuracy"]["ci95"] if run in evals and m in evals[run]["models"] else None))
        rows.append({"run": run, "model": NAMES.get(m, m), "classes": len(man["labels"]), "features": man["n_features"],
                     "subjects": man["n_subjects"], "trials": man["n_samples"], "accuracy": c["accuracy"],
                     "acc_ci95_low": ci[0] if ci else np.nan, "acc_ci95_high": ci[1] if ci else np.nan,
                     "macro_f1": c["macro_f1"], "kappa_quadratic": c["kappa_quadratic"], "mae": c["mae"],
                     "auc": mcmp.get(run, {}).get("models", {}).get(m, {}).get("auc", np.nan), "note": man["note"]})
summary = pd.DataFrame(rows)
summary.to_csv(REPORTS_DIR / "summary.csv", index=False)
write_json(REPORTS_DIR / "summary.json", {"versions": versions, "rows": rows})
print(f"wrote {REPORTS_DIR / 'summary.md'} ({len(L)} lines) and summary.csv ({len(summary)} rows)")
print("\n".join(L[:30]))

wrote D:\HGB\Final_pipeline\outputs\reports\summary.md (485 lines) and summary.csv (180 rows)
# BioVid Part B — results summary for the paper

Generated by Step 11 of `biovid_partB_pipeline.ipynb` from the files in `outputs/`; every number below is read from those files. Accuracy is the proportion of trials classified correctly. Chance is 0.20 (5 classes), 0.25 (4 classes) and 0.50 (2 balanced classes). 95% CIs come from 2000 subject-level bootstrap resamples; p values are two-sided paired bootstrap tests on the same resamples.

## 1. Headline numbers

- **5-class, 169 features (no facial similarity)** (86 subjects, 8426 trials, chance 0.20): XGBoost accuracy 0.360 [0.336, 0.385], macro-F1 0.354, quadratic kappa 0.451; HGB accuracy 0.351 [0.327, 0.376], macro-F1 0.346, quadratic kappa 0.431.
- **4-class BL1/PA2/PA3/PA4, 175 features** (86 subjects, 6755 trials, chance 0.25): XGBoost accuracy 0.440 [0.413, 0.468], macro-F1 0.437, quadratic kappa 0.487; HGB accuracy 0.435 [0.406, 0.464],